# Pipeline OCR — Partie 1 V14.0 — Rotation & Shift Safe

Base : V13.8.7 TTR-7 Fast Structural. Contrat RAW `DOM_EXTRACTION_V1` et **99 champs inchangés** (hash identique).

Changements V14 (qualité d'abord) :
1. **Orientation 0/90/180/270 généralisée** : détecteur déterministe quart-de-tour (Hough, validé 24/24 sur nos dossiers) + appel VLM batché ; rotation physique de l'image **avant gate et extraction** ; re-classification unique des pages tournées restées `AUTRE`/basse confiance (comble le trou de couverture V13.8.7 où un TTR tourné mal classé n'était jamais redressé).
2. **Deskew petite inclinaison réellement câblé** (la brique V13.2 existait mais n'était jamais appelée), appliqué après rotation ; la même géométrie est rejouée sur les rendus HD initiaux et recoveries.
3. **Décalage vertical libellé/valeur (TTR + DOM)** : règle structurelle « ordre vertical des valeurs » renforcée dans le prompt DOM ; recovery déclenché aussi par une **anomalie sémantique sur champ critique** (filet de sécurité anti-décalage).
4. **Permis TTR = structure complète `NN-NNNNNNNN / NN-NN-NNNNNN`** (avec le « / »), aligné sur l'exigence métier ; validateur et prompt TTR7 corrigés en conséquence.
5. **Correction d'audit TTR** : fill rate, champs critiques, statut et confiance calculés sur les **7 champs actifs** du profil TTR7 (les 14 champs inactifs restent à `null` dans `raw_data` sans pénaliser le statut).
6. Nettoyage : suppression du code mort V13.2–V13.4 inatteignable, tests de régression réparés, tokens d'orientation comptabilisés, `trigger_fields` du recovery TTR limités aux champs en échec.


## FINAL V14 — choix d'architecture

Principes inchangés : aucune normalisation métier en Partie 1 ; classification VLM par page ; décalage vertical global pris en compte ; recovery page entière 2400 px uniquement si nécessaire ; `null` plutôt qu'une valeur inventée.

Nouveauté d'ordonnancement :
```
PDF → rendu → classification → orientation (déterministe + VLM) → rotation physique
→ re-classification si page tournée douteuse → deskew → gate réglementaire
→ extraction (TTR7 HD / STANDARD) → recovery 2400 si nécessaire → JSON RAW
```


## 1. Dépendances

In [ ]:
# Si nécessaire sur un environnement neuf Domino :
# %pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow pandas psutil
#
# IMPORTANT : aucune dépendance flash-attn n'est requise ni utilisée.


## 2. Imports

In [ ]:
import gc
import hashlib
import json
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import cv2
import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

print('✅ Imports OK')
print('Python      :', sys.version.split()[0])
print('Torch       :', torch.__version__)
print('CUDA dispo :', torch.cuda.is_available())
print('GPU        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun')


## 3. Configuration V14

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------- Contrat de données (INCHANGÉ) ----------
SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
PIPELINE_VERSION = 'GENERIC_V14_0_PART1_ROTATION_SHIFT_SAFE'
FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'

# ---------- Test ----------
MAX_PDFS = None     # production: tous les PDF; mettre 10 uniquement pour un test
RESUME = True       # un JSON d'une version précédente n'est PAS repris : pipeline_version différent

# ---------- Images ----------
PDF_ZOOM = 2.0
IMAGE_MAX_SIZE = 1400
IMAGE_MAX_SIZE_CLASSIFICATION = 1100
PDF_ZOOM_HAUTE_DEF = 4
IMAGE_MAX_SIZE_HAUTE_DEF = 1800
IMAGE_MAX_SIZE_RECOVERY_2 = 2400
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2400 * 32 * 32
# NOTE V14 : avec MAX_PIXELS = 2400*32*32, l'encodeur vision plafonne chaque image
# à ~2,46 MP. Une page A4 rendue à 2400 px (~4,1 MP) est réduite à ~2,46 MP par le
# processor : le recovery 2400 n'apporte que ~+3,6 % de résolution linéaire vs 1800 px
# (c'est une RELECTURE avec prompt renforcé, pas un vrai gain HD). Augmenter
# MAX_PIXELS serait une décision qualité/coût à benchmarker séparément.

CLASSIFICATION_THRESHOLD = 0.90
CLASSIFICATION_RETRY_ON_AUTRE = True
CLASSIFICATION_RETRY_LOW_CONFIDENCE = True
CLASSIFICATION_HARD_MIN_CONFIDENCE = 0.90
BLOCK_EXTRACTION_ON_CLASSIFICATION_CONFLICT = True

# ---------- V14 : orientation & inclinaison ----------
# ORIENTATION_MODE :
#   'VLM_ALL' (défaut, qualité maximale) : appel orientation VLM batché pour toute
#       page extractible (couvre 0/90/180/270, y compris 180° que le déterministe
#       ne peut pas distinguer de 0°).
#   'HYBRID' : le détecteur déterministe quart-de-tour épargne l'appel VLM aux
#       pages nettement droites non TTR ; VLM conservé pour TTR, QUARTER/UNKNOWN
#       et pages AUTRE / basse confiance. /!\ un 180° sur DOM/CTR/CTS "droit"
#       n'est pas détecté en HYBRID.
ORIENTATION_MODE = 'VLM_ALL'
ORIENTATION_QUARTER_MIN_MARGIN = 0.20
ORIENTATION_MAX_NEW_TOKENS = 80
DESKEW_ENABLED = True
DESKEW_MIN_ABS_DEG = 0.35
DESKEW_MAX_ABS_DEG = 7.0
DESKEW_MIN_LINES = 6

# ---------- Batch ----------
GPU_BATCH_SIZE_CLASSIFICATION = 16
GPU_BATCH_SIZE_EXTRACTION_STANDARD = 4
GPU_BATCH_SIZE_EXTRACTION_HD = 2
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
MAX_NEW_TOKENS_RECOVERY = 1900

# ---------- Budget génération par type (garde-fou ; EOS arrive avant) ----------
MAX_NEW_TOKENS_BY_DOC = {
    'ENGAGEMENT_DOMICILIATION': 900,
    'CONTRAT_TRAVAIL': 1300,
    'CONTRAT_SPECIFIQUE': 1500,
    'TITRE_TRAVAIL': 1600,
    'PERMIS_TRAVAIL_COUVERTURE': 400,
}
MAX_NEW_TOKENS_RECOVERY_BY_DOC = {
    'ENGAGEMENT_DOMICILIATION': 1100,
    'CONTRAT_TRAVAIL': 1400,
    'CONTRAT_SPECIFIQUE': 1600,
    'TITRE_TRAVAIL': 1700,
    'PERMIS_TRAVAIL_COUVERTURE': 500,
}

# Sortie Qwen compacte : les 99 champs finaux restent CANONIQUES dans raw_data.
COMPACT_OUTPUT_ENABLED = True

# ---------- Extraction / recovery page entière ----------
SEUIL_REMPLISSAGE_MIN = 0.40
ENABLE_PAGE_RECOVERY = True
RECOVERY_ON_CRITICAL_MISSING = True
# V14 : une anomalie sémantique sur un champ CRITIQUE (ex. date dans un champ
# salaire après décalage vertical) déclenche aussi le recovery. C'est le filet
# de sécurité des pages à valeurs décalées (DOM notamment) : sans lui, un champ
# critique rempli avec une valeur décalée mais "présente" ne déclenchait rien.
RECOVERY_ON_CRITICAL_SEMANTIC_MISMATCH = True
RECOVERY_ON_COHERENCE_ERROR = False  # cohérence métier réservée à la Partie 2
RECOVERY_ON_LOW_FILL = True

# ---------- Confidence opérationnelle ----------
CONFIDENCE_METHOD = 'OPERATIONAL_CONSENSUS_SEMANTIC_V1'
CONFIDENCE_HIGH = 90
CONFIDENCE_MEDIUM = 75

# ---------- Audit / transparence des appels Qwen ----------
STORE_QWEN_RAW_TEXT_IN_ATTEMPTS = True
STORE_PARSED_DATA_IN_ATTEMPTS = False
PRINT_CALL_DIAGNOSTICS = True

# ---------- Fichiers ----------
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_ROOT = Path('/mnt/data/document_pipeline/V14_0')
RAW_ROOT = OUTPUT_ROOT / '01_extraction_raw'
JSON_DIR = RAW_ROOT / 'json_dossiers'
LOG_PATH = RAW_ROOT / 'pipeline_extraction_v14_0.log'
MANIFEST_PATH = RAW_ROOT / 'extraction_manifest_v14_0.json'
INDEX_CSV_PATH = RAW_ROOT / 'extraction_index_v14_0.csv'

INPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
if MAX_PDFS is not None:
    pdfs = pdfs[:int(MAX_PDFS)]

print('Pipeline       :', PIPELINE_VERSION)
print('Schema         :', SCHEMA_VERSION)
print('Schema hash    :', FIELD_SCHEMA_HASH[:16] + '…')
print('PDFs sélectionnés :', len(pdfs))
print('Entrée         :', INPUT_DIR)
print('Sortie RAW     :', JSON_DIR)
print('Orientation    :', ORIENTATION_MODE, '| deskew :', DESKEW_ENABLED)
print('Recovery page  : UNIQUE', IMAGE_MAX_SIZE_RECOVERY_2, 'px')
print('Couverture PTR : extraction Qwen DESACTIVEE')
print('Confidence     :', CONFIDENCE_METHOD)


## 4. Chargement Qwen — sans Flash-Attention

In [ ]:
if DEVICE != 'cuda':
    raise RuntimeError('Ce pipeline nécessite un GPU CUDA.')

torch.backends.cuda.matmul.allow_tf32 = True

print('Chargement du processor...')
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'left'

print('Chargement du modèle FP8...')
print('ℹ️ V9 : aucune dépendance flash-attn / flash_attention_2.')
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

# Chargement volontairement aligné sur la V7.2/V8.1 qui fonctionne sur Domino.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')
print(f'VRAM allouée : {torch.cuda.memory_allocated()/1e9:.2f} GB')
_device_map = getattr(model, 'hf_device_map', None)
if _device_map:
    print('Device map    :', _device_map)
    _offload = [v for v in _device_map.values() if str(v).lower() in {'cpu','disk'}]
    if _offload:
        print('⚠️ Offload CPU/disk détecté : inférence potentiellement ralentie.')


## 5. Utilitaires PDF / image / JSON

In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert('L'))
    return float((arr > 245).sum() / arr.size)


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f'PDF absent ou vide : {path}')
    pages=[]
    doc=fitz.open(str(path))
    try:
        matrix=fitz.Matrix(zoom, zoom)
        for i in range(doc.page_count):
            page=doc.load_page(i)
            pix=page.get_pixmap(matrix=matrix, alpha=False)
            img=Image.frombytes('RGB',(pix.width,pix.height),pix.samples)
            img=resize_image(img)
            pages.append({
                'index':i, 'page_num':i+1, 'image':img,
                'width':img.width, 'height':img.height,
                'white_ratio':round(white_ratio(img),6),
            })
    finally:
        doc.close()
    return pages


def parse_json_response(text):
    if not text:
        return {}
    clean=str(text).strip()
    clean=re.sub(r'^```(?:json)?','',clean,flags=re.I).strip()
    clean=re.sub(r'```$','',clean).strip()
    match=re.search(r'\{.*\}',clean,flags=re.S)
    if not match:
        return {}
    candidate=match.group(0)
    for attempt in [candidate, re.sub(r',\s*([}\]])',r'\1',candidate)]:
        try:
            obj=json.loads(attempt)
            return obj if isinstance(obj,dict) else {}
        except Exception:
            pass
    return {}


def log(message):
    line=f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH,'a',encoding='utf-8') as f:
        f.write(line+'\n')


def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    largeur, hauteur=image.size
    x0=int(max(0,min(1,gauche))*largeur); x1=int(max(0,min(1,droite))*largeur)
    y0=int(max(0,min(1,haut))*hauteur); y1=int(max(0,min(1,bas))*hauteur)
    if x1<=x0 or y1<=y0:
        return image
    return image.crop((x0,y0,x1,y1))


def render_page_region(pdf_path, page_index, zoom=PDF_ZOOM_HAUTE_DEF,
                       max_side=IMAGE_MAX_SIZE_HAUTE_DEF, crop=None):
    doc=fitz.open(str(pdf_path))
    try:
        page=doc.load_page(int(page_index))
        pix=page.get_pixmap(matrix=fitz.Matrix(zoom,zoom),alpha=False)
        img=Image.frombytes('RGB',(pix.width,pix.height),pix.samples)
    finally:
        doc.close()
    if crop:
        img=crop_region(img,*crop)
    return resize_image(img,max_side=max_side)


def image_for_classification(image):
    return resize_image(image,max_side=IMAGE_MAX_SIZE_CLASSIFICATION)


def is_missing_raw(value):
    # Sert uniquement à décider si Qwen doit relire un champ.
    # La valeur stockée dans raw_data n'est jamais normalisée par cette fonction.
    if value is None:
        return True
    if isinstance(value,str):
        t=value.strip()
        return (not t) or t.upper() in {'NULL','NONE','N/A','NA','ILLISIBLE','NON LISIBLE'}
    return False


def taux_remplissage(data,champs_attendus):
    if not champs_attendus:
        return 1.0
    n=sum(1 for c in champs_attendus if not is_missing_raw((data or {}).get(c)))
    return round(n/len(champs_attendus),4)

print('✅ Utilitaires V9 RAW OK')


## 6. Orientation & deskew (V14)

- `detect_quarter_turn` : déterministe, 0 token. Distingue {0/180} de {90/270} par
  lignes longues (Hough) sur la zone de contenu. Ne tranche JAMAIS entre 0 et 180
  ni entre 90 et 270 (projection identique) : c'est le rôle du VLM.
- `PROMPT_ORIENTATION` + `_parse_orientation` : appel VLM batché, 80 tokens max.
- `resolve_orientations` : applique la rotation physique à l'image stockée,
  re-classe une fois les pages tournées restées douteuses, puis deskew.


In [ ]:
# =====================================================================
# V14 — ORIENTATION / DESKEW AVANT EXTRACTION
# =====================================================================
# Le deskew corrige les petites inclinaisons sans consommer de tokens.
# La rotation 90/180/270 est décidée par le VLM (mode VLM_ALL) ou par
# hybride déterministe+VLM (mode HYBRID), puis appliquée physiquement.
# Aucune rotation n'est jamais faite sur la seule heuristique fragile :
# le détecteur déterministe ne sert qu'à épargner des appels et à auditer.

def _pil_to_cv_gray(img):
    arr=np.asarray(img.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

def estimate_small_skew_deg(img):
    """
    Retourne l'angle dominant des lignes quasi horizontales.
    Valeur >0 : contenu incliné dans le sens anti-horaire.
    None : preuve insuffisante.
    """
    gray=_pil_to_cv_gray(img)
    h,w=gray.shape[:2]
    scale=min(1.0, 1600.0/max(h,w))
    if scale < 1.0:
        gray=cv2.resize(gray, (int(w*scale),int(h*scale)), interpolation=cv2.INTER_AREA)

    blur=cv2.GaussianBlur(gray,(3,3),0)
    edges=cv2.Canny(blur,50,150,apertureSize=3)
    min_len=max(80, int(gray.shape[1]*0.12))
    lines=cv2.HoughLinesP(edges,1,np.pi/1800,threshold=60,
                          minLineLength=min_len,maxLineGap=20)
    if lines is None:
        return None, 0

    angles=[]
    for ln in lines[:,0,:]:
        x1,y1,x2,y2=map(float,ln)
        dx=x2-x1; dy=y2-y1
        if abs(dx)<1:
            continue
        a=np.degrees(np.arctan2(dy,dx))
        if -DESKEW_MAX_ABS_DEG <= a <= DESKEW_MAX_ABS_DEG:
            angles.append(a)

    if len(angles) < DESKEW_MIN_LINES:
        return None, len(angles)

    med=float(np.median(angles))
    mad=float(np.median(np.abs(np.asarray(angles)-med))) if angles else 999.0
    if mad > 1.8:
        return None, len(angles)
    return med, len(angles)

def rotate_pil_expand_white(img, angle_deg):
    arr=np.asarray(img.convert("RGB"))
    h,w=arr.shape[:2]
    center=(w/2.0,h/2.0)
    M=cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    cos=abs(M[0,0]); sin=abs(M[0,1])
    nw=int(h*sin+w*cos); nh=int(h*cos+w*sin)
    M[0,2]+=nw/2-center[0]
    M[1,2]+=nh/2-center[1]
    rot=cv2.warpAffine(arr,M,(nw,nh),flags=cv2.INTER_CUBIC,
                       borderMode=cv2.BORDER_CONSTANT,borderValue=(255,255,255))
    return Image.fromarray(rot)

def deskew_for_vlm(img):
    meta={
        "deskew_enabled": bool(DESKEW_ENABLED),
        "deskew_applied": False,
        "deskew_detected_angle_deg": None,
        "deskew_correction_angle_deg": 0.0,
        "deskew_evidence_lines": 0,
    }
    if not DESKEW_ENABLED:
        return img,meta

    angle,nlines=estimate_small_skew_deg(img)
    meta["deskew_evidence_lines"]=int(nlines)
    if angle is None:
        return img,meta

    meta["deskew_detected_angle_deg"]=round(float(angle),3)
    if DESKEW_MIN_ABS_DEG <= abs(angle) <= DESKEW_MAX_ABS_DEG:
        corr=-float(angle)
        out=rotate_pil_expand_white(img,corr)
        meta["deskew_applied"]=True
        meta["deskew_correction_angle_deg"]=round(corr,3)
        return out,meta
    return img,meta

def detect_quarter_turn(img, min_ink=0.001, margin_need=None):
    """
    Distingue {0/180} ('UPRIGHT') de {90/270} ('QUARTER') par les lignes longues
    du document (Hough), pondérées par longueur, sur la zone de contenu.
    Ne prétend JAMAIS choisir entre 0 et 180, ni entre 90 et 270.
    'UNKNOWN' si encre insuffisante ou marge faible.
    """
    if margin_need is None:
        margin_need=ORIENTATION_QUARTER_MIN_MARGIN
    g=_pil_to_cv_gray(img)
    h,w=g.shape
    s=min(1.0,900.0/max(h,w))
    if s<1: g=cv2.resize(g,(int(w*s),int(h*s)),interpolation=cv2.INTER_AREA)
    ink_mask=(g<200).astype(np.uint8)
    if ink_mask.mean()<min_ink:
        return 'UNKNOWN',0.0
    ys,xs=np.where(ink_mask)
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max()
    pad=8
    g=g[max(0,y0-pad):min(g.shape[0],y1+pad), max(0,x0-pad):min(g.shape[1],x1+pad)]
    blur=cv2.GaussianBlur(g,(3,3),0)
    edges=cv2.Canny(blur,50,150,apertureSize=3)
    lines=cv2.HoughLinesP(edges,1,np.pi/1800,threshold=50,
                          minLineLength=max(60,int(g.shape[1]*0.15)),maxLineGap=25)
    if lines is None or len(lines)<4:
        return 'UNKNOWN',0.0
    horiz=vert=0.0
    for ln in lines[:,0,:]:
        x1l,y1l,x2l,y2l=map(float,ln)
        dx,dy=x2l-x1l,y2l-y1l
        L=(dx*dx+dy*dy)**0.5
        if L<1: continue
        a=abs(np.degrees(np.arctan2(dy,dx)))
        a=min(a,180-a)
        if a<=20: horiz+=L
        elif a>=70: vert+=L
    m=max(horiz,vert)
    if m<=0:
        return 'UNKNOWN',0.0
    margin=abs(horiz-vert)/m
    if margin<margin_need:
        return 'UNKNOWN',round(float(margin),3)
    return ('UPRIGHT' if horiz>=vert else 'QUARTER'), round(float(margin),3)

def _rotate_clockwise(img, angle):
    a = int(angle or 0) % 360
    if a == 0 or a not in (90,180,270):
        return img
    # PIL: angle positif = anti-horaire.
    return img.rotate(-a, expand=True)

PROMPT_ORIENTATION = r"""
Cette page provient d'un dossier administratif (formulaire, contrat ou
titre/permis de travail, en français et/ou arabe).
Détermine la rotation HORAIRE qu'il faut appliquer à l'image pour remettre le
document à l'endroit et lire normalement son titre et ses libellés.
Réponds STRICTEMENT:
{"rotation_clockwise":0}
Valeurs autorisées uniquement: 0, 90, 180, 270.
Ne lis aucun champ métier et n'invente aucune donnée.
"""

def _parse_orientation(text):
    obj = parse_json_response(text)
    try:
        a = int((obj or {}).get("rotation_clockwise", 0))
    except Exception:
        a = 0
    return a if a in (0,90,180,270) else 0

_ORIENTATION_KEYS = ('rotation_clockwise','rotation_applied',
    'orientation_quarter_detect','orientation_quarter_margin',
    'orientation_raw_text','orientation_tokens_in','orientation_tokens_out',
    'orientation_elapsed_s','quality_flags')

def resolve_orientations(records):
    """
    V14 : orientation AVANT gate/extraction, pour toutes les pages extractibles.
    1) détection déterministe quart-de-tour (audit + cross-check) ;
    2) appel VLM orientation batché selon ORIENTATION_MODE ;
    3) rotation physique de l'image stockée ;
    4) re-classification unique des pages tournées restées AUTRE / basse confiance ;
    5) deskew petite inclinaison (déterministe) sur l'image redressée.
    """
    for r in records:
        det,margin=detect_quarter_turn(r['image'])
        r['orientation_quarter_detect']=det
        r['orientation_quarter_margin']=margin

    def _needs_vlm(r):
        if r.get('doc_type')=='PERMIS_TRAVAIL_COUVERTURE':
            return False
        det=r.get('orientation_quarter_detect')
        if ORIENTATION_MODE=='VLM_ALL':
            return r.get('doc_type') in PROMPTS_EXTRACTION or r.get('doc_type')=='AUTRE'
        return (r.get('doc_type')=='TITRE_TRAVAIL'
                or det in ('QUARTER','UNKNOWN')
                or r.get('doc_type')=='AUTRE'
                or float(r.get('classification_confidence',0) or 0)<CLASSIFICATION_HARD_MIN_CONFIDENCE)

    targets=[r for r in records if _needs_vlm(r)]
    for start in range(0,len(targets),GPU_BATCH_SIZE_CLASSIFICATION):
        batch=targets[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
        outs=ask_batch(PROMPT_ORIENTATION,[image_for_classification(r['image']) for r in batch],
                       ORIENTATION_MAX_NEW_TOKENS)
        _shared=sum(float(o.get('elapsed_s',0) or 0) for o in outs)
        for r,o in zip(batch,outs):
            angle=_parse_orientation(o.get('text',''))
            r['rotation_clockwise']=angle
            r['orientation_raw_text']=o.get('text')
            r['orientation_tokens_in']=int(o.get('tokens_in',0) or 0)
            r['orientation_tokens_out']=int(o.get('tokens_out',0) or 0)
            r['orientation_elapsed_s']=float(o.get('elapsed_s',0) or 0)
            # Cross-check déterministe : {0,180} vs {90,270}
            det=r.get('orientation_quarter_detect')
            if det in ('UPRIGHT','QUARTER') and ((angle in (90,270)) != (det=='QUARTER')):
                r.setdefault('quality_flags',[]).append('ORIENTATION_VLM_VS_DETERMINISTIC_CONFLICT')
            page_pevent(r.get('page_num'), r.get('doc_type'), 'ORIENTATION','VLM_ORIENTATION',
                        _shared, len(batch), o.get('tokens_in',0), o.get('tokens_out',0),
                        note='TEMPS_BATCH_PARTAGE' if len(batch)>1 else 'TEMPS_PAGE_DIRECT')

    for r in records:
        r.setdefault('rotation_clockwise',0)
        r.setdefault('orientation_raw_text',None)
        r.setdefault('orientation_tokens_in',0)
        r.setdefault('orientation_tokens_out',0)
        r.setdefault('orientation_elapsed_s',0.0)
        a=r.get('rotation_clockwise',0) or 0
        r['rotation_applied']=bool(a)
        if a:
            r['image']=_rotate_clockwise(r['image'],a)
            r['width'],r['height']=r['image'].size

    # Pages tournées dont la classification reste douteuse : une seule re-classification
    # sur l'image redressée (comble le cas "TTR tourné classé AUTRE").
    retry=[r for r in records
           if r.get('rotation_applied')
           and (r.get('doc_type')=='AUTRE'
                or float(r.get('classification_confidence',0) or 0)<CLASSIFICATION_HARD_MIN_CONFIDENCE)]
    for start in range(0,len(retry),GPU_BATCH_SIZE_CLASSIFICATION):
        batch=retry[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
        outs=ask_batch(PROMPT_CLASSIFICATION,[r['image'] for r in batch],MAX_NEW_TOKENS_CLASSIFICATION)
        _shared=sum(float(o.get('elapsed_s',0) or 0) for o in outs)
        for r,out in zip(batch,outs):
            parsed=parse_json_response(out['text'])
            page_pevent(r.get('page_num'), parsed.get('type_document') or r.get('doc_type'),
                        'CLASSIFICATION_RETRY','POST_ROTATION_RETRY', _shared, len(batch),
                        out.get('tokens_in',0), out.get('tokens_out',0))
            attempt={'strategy':'POST_ROTATION_RETRY','raw_text':out.get('text'),'parsed':parsed,
                     'tokens_in':out.get('tokens_in',0),'tokens_out':out.get('tokens_out',0),
                     'elapsed_s':out.get('elapsed_s',0)}
            new=_new_record_from_page({'page_num':r['page_num'],'width':r['width'],'height':r['height'],
                                       'white_ratio':r['white_ratio'],'image':r['image']},parsed,out)
            attempts=list(r.get('classification_attempts') or [])+[attempt]
            keep={k:r.get(k) for k in _ORIENTATION_KEYS}
            new['classification_attempts']=attempts
            new['classification_tokens_in']=sum(int(a.get('tokens_in',0) or 0) for a in attempts)
            new['classification_tokens_out']=sum(int(a.get('tokens_out',0) or 0) for a in attempts)
            new['classification_elapsed_s']=round(sum(float(a.get('elapsed_s',0) or 0) for a in attempts),3)
            new['classification_raw_text']='\n\n'.join(f"[{a['strategy']}] {a.get('raw_text','')}" for a in attempts)
            new['classification_retry_fullres']=True
            r.clear(); r.update(new)
            for k,v in keep.items():
                if v is not None: r[k]=v

    for r in records:
        img,meta=deskew_for_vlm(r['image'])
        r['image']=img
        r.update(meta)
    return records

print('✅ V14 orientation + deskew prêts (déterministe + VLM batché)')


## 7. Profiler V14 — temps réel par étape ET par page

In [ ]:

# =====================================================================
# V13.7 PROFILER — temps réel par étape ET par page
# =====================================================================
from contextlib import contextmanager

PROFILER_ENABLED = True
PROFILER_EVENTS = []
PAGE_PROFILER_ROWS = []
PROFILER_T0 = None

def _pnow():
    return time.perf_counter()

def _psync():
    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass

def _pelapsed():
    return 0.0 if PROFILER_T0 is None else _pnow() - PROFILER_T0

def _pprint(msg):
    if PROFILER_ENABLED:
        print(f"[PROF {_pelapsed():8.3f}s] {msg}", flush=True)

def profiler_reset():
    global PROFILER_EVENTS, PAGE_PROFILER_ROWS, PROFILER_T0
    PROFILER_EVENTS = []
    PAGE_PROFILER_ROWS = []
    PROFILER_T0 = _pnow()
    _pprint("START DOSSIER")

def pevent(stage, seconds, **meta):
    PROFILER_EVENTS.append({
        "stage": stage,
        "elapsed_s": round(float(seconds), 6),
        **meta
    })

def page_pevent(page_num, doc_type, stage, strategy,
                shared_batch_s, batch_size=1,
                tokens_in=0, tokens_out=0, note=None):
    """Le temps exact d'un batch GPU est partagé entre ses pages.
    allocated_s = temps batch / nombre d'éléments, uniquement pour obtenir
    une imputation additive par page sans prétendre à un temps GPU isolé.
    """
    batch_size = max(1, int(batch_size or 1))
    row = {
        "page": page_num,
        "doc_type": doc_type,
        "stage": stage,
        "strategy": strategy,
        "batch_size": batch_size,
        "shared_batch_s": round(float(shared_batch_s), 3),
        "allocated_s": round(float(shared_batch_s) / batch_size, 3),
        "tokens_in": int(tokens_in or 0),
        "tokens_out": int(tokens_out or 0),
        "note": note or "",
    }
    PAGE_PROFILER_ROWS.append(row)
    _pprint(
        f"PAGE {page_num} | {doc_type} | {stage} | {strategy} | "
        f"batch={batch_size} temps_partage={row['shared_batch_s']:.3f}s "
        f"temps_impute={row['allocated_s']:.3f}s | "
        f"tokens={row['tokens_in']}+{row['tokens_out']}"
    )

@contextmanager
def pstage(stage, **meta):
    _psync()
    t = _pnow()
    _pprint(f"START {stage} | {meta}" if meta else f"START {stage}")
    try:
        yield
    finally:
        _psync()
        d = _pnow() - t
        pevent(stage, d, **meta)
        _pprint(f"END   {stage} | {d:.3f}s")

def profiler_finish(root):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    if PROFILER_EVENTS:
        df = pd.DataFrame(PROFILER_EVENTS)
        sm = (df.groupby("stage")
                .agg(appels=("stage", "size"), temps_s=("elapsed_s", "sum"))
                .reset_index()
                .sort_values("temps_s", ascending=False))
        total = float(sm.temps_s.sum()) or 1.0
        sm["pct_mesure"] = (sm.temps_s / total * 100).round(1)

        print("\n================ PROFILER GLOBAL =================")
        print(sm.to_string(index=False))
        print("==================================================")

        df.to_csv(root / "profiler_events_v14_0.csv",
                  index=False, encoding="utf-8-sig")
        sm.to_csv(root / "profiler_summary_v14_0.csv",
                  index=False, encoding="utf-8-sig")

    if PAGE_PROFILER_ROWS:
        pdf = pd.DataFrame(PAGE_PROFILER_ROWS)

        print("\n================ DETAIL PAR PAGE =================")
        cols = ["page","doc_type","stage","strategy","batch_size",
                "shared_batch_s","allocated_s","tokens_in","tokens_out","note"]
        print(pdf[cols].to_string(index=False))

        # Temps imputé = métrique additive. Le temps partagé du batch reste visible
        # dans le détail et ne doit pas être additionné page par page.
        ps = (pdf.groupby(["page","doc_type"], dropna=False)
                .agg(
                    qwen_etapes=("stage","size"),
                    temps_impute_s=("allocated_s","sum"),
                    tokens_in=("tokens_in","sum"),
                    tokens_out=("tokens_out","sum"),
                )
                .reset_index()
                .sort_values(["page","doc_type"]))

        print("\n================ RESUME PAR PAGE =================")
        print(ps.to_string(index=False))
        print("NOTE : 'temps_impute_s' répartit le temps d'un batch entre ses pages.")
        print("       'shared_batch_s' dans le détail est le vrai temps GPU du batch.")
        print("==================================================")

        pdf.to_csv(root / "profiler_pages_detail_v14_0.csv",
                   index=False, encoding="utf-8-sig")
        ps.to_csv(root / "profiler_pages_summary_v14_0.csv",
                  index=False, encoding="utf-8-sig")

        print("📊", root / "profiler_pages_detail_v14_0.csv")
        print("📊", root / "profiler_pages_summary_v14_0.csv")


## 8. Inférence GPU batch

In [ ]:
def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def ask_single(prompt, image, max_new_tokens):
    messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    with pstage("CHAT_TEMPLATE",batch=1):
        text_in=apply_template(messages)
    with pstage("PROCESSOR",batch=1):
        inputs=processor(text=[text_in],images=[image],return_tensors='pt')
    with pstage("TRANSFER_GPU",batch=1):
        inputs=inputs.to(DEVICE)
    tin=int(inputs['input_ids'].shape[1])
    _psync(); t=_pnow(); _pprint(f"START MODEL_GENERATE | batch=1 in={tin} max_new={max_new_tokens}")
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
            repetition_penalty=1.0,use_cache=True,pad_token_id=processor.tokenizer.eos_token_id)
    _psync(); gen_s=_pnow()-t
    generated=out[0][inputs['input_ids'].shape[1]:]; tout=int(len(generated))
    pevent("MODEL_GENERATE",gen_s,batch=1,tokens_in=tin,tokens_out=tout,max_new_tokens=max_new_tokens)
    _pprint(f"END   MODEL_GENERATE | {gen_s:.3f}s | in={tin} out={tout}")
    with pstage("DECODE",batch=1,tokens_out=tout):
        text=processor.decode(generated,skip_special_tokens=True,clean_up_tokenization_spaces=True)
    return {'text':text,'tokens_in':tin,'tokens_out':tout,'elapsed_s':round(gen_s,3)}


def ask_batch_mixed(prompts, images, max_new_tokens):
    if not images: return []
    if len(prompts)!=len(images): raise ValueError('prompts et images doivent avoir la même longueur')
    if len(images)==1: return [ask_single(prompts[0],images[0],max_new_tokens)]
    texts_in=[]
    with pstage("CHAT_TEMPLATE",batch=len(images)):
        for prompt,image in zip(prompts,images):
            messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
            texts_in.append(apply_template(messages))
    with pstage("PROCESSOR",batch=len(images)):
        inputs=processor(text=texts_in,images=images,return_tensors='pt',padding=True)
    with pstage("TRANSFER_GPU",batch=len(images)):
        inputs=inputs.to(DEVICE)
    input_width=inputs['input_ids'].shape[1]
    attention_mask=inputs.get('attention_mask')
    total_in=int(attention_mask.sum().item()) if attention_mask is not None else int(input_width*len(images))
    _psync(); t=_pnow(); _pprint(f"START MODEL_GENERATE | batch={len(images)} in={total_in} max_new={max_new_tokens}")
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
            repetition_penalty=1.0,use_cache=True,pad_token_id=processor.tokenizer.eos_token_id)
    _psync(); gen_s=_pnow()-t
    if out.shape[0]!=len(images): raise RuntimeError(f'Réponses VLM incohérentes : {out.shape[0]} sortie(s) pour {len(images)} image(s)')
    results=[]; total_out=0
    with pstage("DECODE",batch=len(images)):
        for i in range(len(images)):
            generated=out[i][input_width:]
            text=processor.decode(generated,skip_special_tokens=True,clean_up_tokenization_spaces=True)
            tin=int(attention_mask[i].sum().item()) if attention_mask is not None else int(input_width)
            tout=int(len(generated)); total_out+=tout
            results.append({'text':text,'tokens_in':tin,'tokens_out':tout,'elapsed_s':round(gen_s/len(images),3)})
    seq_out=[int(x.get('tokens_out',0) or 0) for x in results]
    pevent(
        "MODEL_GENERATE",gen_s,batch=len(images),tokens_in=total_in,
        tokens_out=total_out,sequence_tokens_out=str(seq_out),
        max_new_tokens=max_new_tokens
    )
    _pprint(
        f"END   MODEL_GENERATE | {gen_s:.3f}s | batch={len(images)} "
        f"in={total_in} out={total_out} | seq_out={seq_out}"
    )
    return results


def ask_batch(prompt, images, max_new_tokens):
    """Compatibilité avec la classification V7 : même prompt pour le batch."""
    return ask_batch_mixed([prompt] * len(images), images, max_new_tokens)


def is_cuda_oom(exc):
    text = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or 'out of memory' in text


print('✅ Inférence V9 single / batch multi-prompts OK')


## 9. Prompts — mêmes 99 champs que V8.1

V14 : le prompt DOM est renforcé pour le **décalage vertical global** (règle
d'ordre vertical des valeurs). Le grand prompt TTR historique est remplacé par
un squelette de référence (l'extraction TTR utilise `PROMPT_TTR7_FAST_STRUCTURAL`,
cellule 12) ; la liste des 21 champs TTR est strictement identique (hash préservé).


In [ ]:
PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

RÈGLE DE PRIORITÉ ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « جواز العمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
    - une photographie d'identité ;
    - les libellés « Nom » et « Prénom » suivis de valeurs ;
    - les libellés « Date de naissance » / « Lieu de naissance » ;
    - le libellé « Date d'entrée en Algérie » ;
    - des libellés arabes d'identité (اللقب، الإسم، تاريخ الإزدياد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, identifier d'abord le libellé et la structure du formulaire.
3. IMPORTANT — FORMULAIRES PRÉ-IMPRIMÉS :
   la couche des valeurs peut être décalée verticalement par rapport aux libellés.
   Le décalage peut être VERS LE HAUT ou VERS LE BAS et peut concerner TOUTE LA PAGE.
   Ne jamais associer une valeur à un champ uniquement parce qu'elle est exactement
   sur la même ligne horizontale que le libellé.
4. Utiliser conjointement :
   - le libellé ;
   - l'ordre des champs du formulaire ;
   - les libellés voisins au-dessus et au-dessous ;
   - le type de valeur attendu (date, lieu, nom, montant, référence...) ;
   - la cohérence globale de la page.
5. Un décalage global doit rester cohérent sur la page : ne pas déplacer
   arbitrairement une seule valeur d'un champ vers un autre.
6. Conserver la valeur exactement comme elle apparaît : espaces, ponctuation,
   séparateurs, format de date et format de montant.
7. Ne corrige pas l'orthographe.
8. Ne normalise pas les dates.
9. Ne normalise pas les montants.
10. Ne sépare pas automatiquement le nom et le prénom.
11. Ne complète pas une valeur partiellement lisible.
12. N'utilise aucune valeur provenant d'une autre page.
13. Si le libellé est absent, si la valeur est illisible ou si l'association
    libellé/valeur reste ambiguë malgré la structure de la page : retourne null.
14. N'invente jamais une valeur.
15. Retourne uniquement un objet JSON valide, sans commentaire.
16. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

MISE EN PAGE : sur certains engagements, toutes les valeurs imprimées peuvent être
décalées verticalement de façon uniforme vers le haut ou vers le bas par rapport
aux libellés. Utilise la structure entière du formulaire et le type attendu de chaque
valeur ; ne fais jamais un appariement strict par ligne.

RÈGLE STRUCTURELLE — DÉCALAGE GLOBAL DES VALEURS (PRIORITAIRE) :
Sur certains engagements, TOUTE la couche des valeurs imprimées est décalée
verticalement (vers le haut ou vers le bas), parfois d'une ligne entière ou plus.
La valeur d'un champ peut donc se trouver à la hauteur du libellé PRÉCÉDENT ou
SUIVANT. Méthode obligatoire :
1. Relève la séquence verticale des valeurs imprimées, DE HAUT EN BAS.
2. Associe-les aux libellés dans LE MÊME ORDRE vertical, en vérifiant le TYPE
   attendu : un numéro de contrat n'est pas une durée ; une date n'est pas un
   salaire ; un salaire (montant) n'est pas un pourcentage (%).
3. L'ordre des champs du formulaire est fixe :
   Nom et raison sociale -> N de compte -> Adresse ->
   Numéro du contrat -> Duré du contrat -> Date de début -> Date de fin ->
   Nom employeur -> Adresse employeur -> Salaire net mensuel ->
   Montant part transférable -> Pourcentage -> Montant domicilié en DZD.
4. Ne fais jamais d'appariement par alignement horizontal seul.
5. Si l'association reste ambiguë malgré l'ordre et le type : null.

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L'Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.

- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD
  null si ce cachet est absent de la page.

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».

- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---

Cette ligne peut prendre DEUX formes :

  Forme A (salaire inchangé) :
    « Salaire mensuel de base net : 506,471.38 »

  Forme B (augmentation de salaire) :
    « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.

- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « Le présent contrat a été visé par nous ».
"""

PROMPT_TITRE_TRAVAIL = """
Type : TITRE_TRAVAIL.
V14 : l'extraction de ce type est réalisée par PROMPT_TTR7_FAST_STRUCTURAL
(cellule 12). Ce prompt de référence conserve uniquement le squelette exact
des 21 champs, pour le contrat de schéma (hash) et la cohérence des outils.

{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null,
  "TTR_PHOTO_PRESENTE": null,
  "TTR_CACHET_PRESENT": null
}
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Cette image correspond à la couverture du permis de travail :
titre « جواز العمل / Permis de Travail », extraits de loi et cachet
de la Direction de l'Emploi de la Wilaya.

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null,
  "PTR_WILAYA": null,
  "PTR_CACHET_DIRECTION_EMPLOI_PRESENT": null
}

- PTR_NUMERO_SERIE :
  numéro de série imprimé en bas de la couverture,
  après « N° de Série » ou isolé en bas à gauche.
  null si illisible.

- PTR_WILAYA :
  valeur après « Direction de l'Emploi de la Wilaya de : ».
  null si la ligne est vide.

- PTR_CACHET_DIRECTION_EMPLOI_PRESENT :
  true si un cachet officiel est visible sur cette zone.

Ne pas extraire le contenu des extraits de loi imprimés à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}


def champs_attendus_depuis_prompt(prompt):
    """
    Récupère la liste des clés depuis le squelette JSON du prompt.
    Évite de maintenir une seconde liste qui divergerait des prompts.
    """
    match = re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return []
    bloc = match.group(0)
    try:
        return list(json.loads(bloc).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', bloc)



CHAMPS_ATTENDUS = {
    doc_type: champs_attendus_depuis_prompt(prompt)
    for doc_type, prompt in PROMPTS_EXTRACTION.items()
}

# ---------------------------------------------------------------------
# V13.5 — FORMAT DE SORTIE COMPACT
# ---------------------------------------------------------------------
# Qwen continue de recevoir toutes les règles métier et tous les noms de champs,
# mais il renvoie des clés courtes f01, f02... Python les remappe ensuite vers
# les noms canoniques. Le JSON RAW final conserve donc strictement les 99 champs.
COMPACT_ALIAS_BY_DOC = {
    dt: {field: f"f{i+1:02d}" for i,field in enumerate(fields)}
    for dt,fields in CHAMPS_ATTENDUS.items()
}
COMPACT_FIELD_BY_DOC = {
    dt: {alias: field for field,alias in mapping.items()}
    for dt,mapping in COMPACT_ALIAS_BY_DOC.items()
}

def _replace_first_json_skeleton_with_aliases(prompt, doc_type):
    mapping=COMPACT_ALIAS_BY_DOC.get(doc_type) or {}
    if not mapping:
        return prompt

    alias_skeleton={alias:None for alias in mapping.values()}
    match=re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return prompt

    alias_map_lines="\n".join(
        f"- {alias} = {field}"
        for field,alias in mapping.items()
    )
    compact_json=json.dumps(alias_skeleton,ensure_ascii=False,indent=2)

    result=prompt[:match.start()] + compact_json + prompt[match.end():]
    result += f"""

FORMAT DE SORTIE V13.5 — PRIORITÉ ABSOLUE :
Pour réduire la longueur de génération, retourne les champs métier avec les
ALIASES COURTS ci-dessous. Ne retourne PAS les noms longs des champs.

{alias_map_lines}

Retourne un seul objet JSON valide.
- Chaque alias ci-dessus doit être présent, avec sa valeur ou null.
- Pour TITRE_TRAVAIL uniquement, conserve aussi l'objet d'audit
  "TTR_STRUCTURAL_BLOCK" demandé plus haut.
- N'ajoute aucun commentaire ni texte hors JSON.
"""
    return result

def compact_prompt_for_doc(doc_type, prompt):
    if not COMPACT_OUTPUT_ENABLED:
        return prompt
    return _replace_first_json_skeleton_with_aliases(prompt,doc_type)

def expand_compact_extraction(parsed, doc_type):
    """Remappe f01/f02/... vers les champs canoniques, sans toucher aux métadonnées."""
    if not isinstance(parsed,dict):
        return {}
    reverse=COMPACT_FIELD_BY_DOC.get(doc_type) or {}
    if not reverse:
        return parsed

    out={}
    compact_seen=False
    for k,v in parsed.items():
        if k in reverse:
            out[reverse[k]]=v
            compact_seen=True
        else:
            # Conserver notamment TTR_STRUCTURAL_BLOCK.
            out[k]=v

    # Si Qwen a malgré tout utilisé les noms canoniques, ils restent acceptés.
    return out

# ---------------------------------------------------------------------
# V8.1 - champs critiques et stratégies de relecture
# ---------------------------------------------------------------------

CRITICAL_FIELDS = {
    'ENGAGEMENT_DOMICILIATION': {
        'DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL',
        'DOM_NUMERO_CONTRAT', 'DOM_DATE_DEBUT_CONTRAT',
        'DOM_DATE_FIN_CONTRAT', 'DOM_SALAIRE_NET_MENSUEL',
        'DOM_PART_TRANSFERABLE',
    },
    'CONTRAT_TRAVAIL': {
        'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_NUMERO_PERMIS_TRAVAIL',
        'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS',
        'CTR_SALAIRE_NET',
    },
    'CONTRAT_SPECIFIQUE': {
        'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_NUMERO_PERMIS_TRAVAIL',
        'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS',
        'CTS_SALAIRE_NET', 'CTS_PART_TRANSFERABLE',
    },
    'TITRE_TRAVAIL': {
        'TTR_NUMERO_PERMIS', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN',
        'TTR_EMPLOYEUR', 'TTR_NOM', 'TTR_PRENOM',
        'TTR_DATE_NAISSANCE', 'TTR_NATIONALITE',
        'TTR_DATE_ENTREE_ALGERIE',
    },
    'PERMIS_TRAVAIL_COUVERTURE': {
        'PTR_NUMERO_SERIE',
    },
}

# ---------------------------------------------------------------------
# FINAL V12 — validation sémantique légère (sans normaliser le RAW)
# ---------------------------------------------------------------------
DATE_FIELDS = {
    'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','DOM_DATE_SIGNATURE',
    'CTR_DATE_DEBUT_CONTRAT','CTR_DATE_NAISSANCE','CTR_DATE_DELIVRANCE_PERMIS',
    'CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','CTR_DATE_SIGNATURE',
    'CTS_DATE_DEBUT_CONTRAT','CTS_DATE_NAISSANCE','CTS_DATE_DELIVRANCE_PERMIS',
    'CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','CTS_DATE_DOCUMENT',
    'TTR_DATE_DEBUT','TTR_DATE_FIN','TTR_DATE_DELIVRANCE','TTR_DATE_NAISSANCE',
    'TTR_DATE_ENTREE_ALGERIE',
}

AMOUNT_FIELDS = {
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE','DOM_TAUX_TRANSFERABLE',
    'DOM_MONTANT_TOTAL_DOMICILIE','CTR_SALAIRE_BRUT','CTR_SALAIRE_NET',
    'CTS_LIGNE_SALAIRE_BRUTE','CTS_SALAIRE_NET','CTS_SALAIRE_NET_ANCIEN',
    'CTS_PART_TRANSFERABLE','CTS_PART_PAYABLE_DZD',
}

DURATION_FIELDS = {'DOM_DUREE_CONTRAT_MOIS','CTR_DUREE_MOIS','CTS_DUREE_MOIS','TTR_DUREE'}
BOOLEAN_FIELDS = {
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE','CTR_SIGNATURE_EMPLOYEUR_PRESENTE','CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE','CTS_SIGNATURE_TRAVAILLEUR_PRESENTE',
    'CTS_SIGNATURE_EMPLOYEUR_PRESENTE','CTS_CACHET_EMPLOYEUR_PRESENT','CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE','TTR_CACHET_PRESENT','PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
PERMIT_REFERENCE_FIELDS = {'TTR_NUMERO_PERMIS','CTR_NUMERO_PERMIS_TRAVAIL','CTS_NUMERO_PERMIS_TRAVAIL','PTR_NUMERO_SERIE'}

# Pour ces champs, une valeur qui ressemble clairement à une date indique souvent
# un décalage de formulaire (ex. LIEU_TRAVAIL = 14/08/2025).
TEXT_FIELDS_REJECT_DATE = {
    'DOM_NOM_RAISON_SOCIAL_CLIENT','DOM_ADRESSE_CLIENT','DOM_AGENCE_DOMICILIATAIRE',
    'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR','DOM_ADRESSE_EMPLOYEUR',
    'CTR_EMPLOYEUR','CTR_ACTIVITE_EMPLOYEUR','CTR_POSTE','CTR_NOM_PRENOM_TRAVAILLEUR',
    'CTR_PERE_NOM_PRENOM','CTR_MERE_NOM_PRENOM','CTR_NATIONALITE','CTR_LIEU_PAYS_NAISSANCE',
    'CTR_ADRESSE_ALGERIE','CTR_QUALIFICATION',
    'CTS_EMPLOYEUR','CTS_ACTIVITE_EMPLOYEUR','CTS_POSTE','CTS_NOM_PRENOM_TRAVAILLEUR',
    'CTS_PERE_NOM_PRENOM','CTS_MERE_NOM_PRENOM','CTS_NATIONALITE','CTS_LIEU_PAYS_NAISSANCE',
    'CTS_ADRESSE_ALGERIE','CTS_QUALIFICATION',
    'TTR_POSTE','TTR_LIEU_TRAVAIL','TTR_EMPLOYEUR','TTR_ADRESSE_EMPLOYEUR','TTR_FAIT_A',
    'TTR_NOM','TTR_PRENOM','TTR_LIEU_NAISSANCE','TTR_PAYS','TTR_NATIONALITE','TTR_QUALIFICATION',
    'PTR_WILAYA',
}

_MONTH_WORDS = ('JANVIER','FEVRIER','FÉVRIER','MARS','AVRIL','MAI','JUIN','JUILLET',
                'AOUT','AOÛT','SEPTEMBRE','OCTOBRE','NOVEMBRE','DECEMBRE','DÉCEMBRE')


def _clean_scalar_text(value):
    return '' if value is None else str(value).strip()


def looks_like_date(value):
    s=_clean_scalar_text(value).upper()
    if not s:
        return False
    # Formats numériques usuels. On ne normalise rien ; on vérifie seulement la forme.
    if re.search(r'\b(?:0?[1-9]|[12]\d|3[01])[./-](?:0?[1-9]|1[0-2])[./-](?:19|20)\d{2}\b', s):
        return True
    if re.search(r'\b(?:19|20)\d{2}[./-](?:0?[1-9]|1[0-2])[./-](?:0?[1-9]|[12]\d|3[01])\b', s):
        return True
    if any(m in s for m in _MONTH_WORDS) and re.search(r'\b(?:19|20)\d{2}\b',s):
        return True
    return False


def parse_date_for_coherence(value):
    """Parse seulement pour contrôle de cohérence ; ne modifie jamais le RAW."""
    s=_clean_scalar_text(value)
    if not s:
        return None
    m=re.search(r'\b(\d{1,2})[./-](\d{1,2})[./-]((?:19|20)\d{2})\b',s)
    if m:
        try: return datetime(int(m.group(3)),int(m.group(2)),int(m.group(1)))
        except Exception: return None
    m=re.search(r'\b((?:19|20)\d{2})[./-](\d{1,2})[./-](\d{1,2})\b',s)
    if m:
        try: return datetime(int(m.group(1)),int(m.group(2)),int(m.group(3)))
        except Exception: return None
    return None


def semantic_issue_for_field(field,value):
    if is_missing_raw(value):
        return None
    s=_clean_scalar_text(value)
    digits=re.sub(r'\D','',s)

    if field in DATE_FIELDS and not looks_like_date(s):
        return 'EXPECTED_DATE'
    if field in AMOUNT_FIELDS and len(digits)==0:
        return 'EXPECTED_NUMERIC_VALUE'
    if field in DURATION_FIELDS:
        # Une durée doit contenir une unité temporelle explicite.
        # Evite le faux positif réel : "TAGE / LAMINOR 2" contient un chiffre
        # mais correspond à une continuation du poste, pas à une durée.
        dur=s.upper()
        if not re.search(r'\b\d{1,3}\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES|MOIS|JOUR|JOURS)\b', dur):
            return 'EXPECTED_DURATION'
    if field == 'DOM_COMPTE_LOCAL' and len(digits) < 10:
        return 'EXPECTED_ACCOUNT_NUMBER'
    if field in PERMIT_REFERENCE_FIELDS and len(re.sub(r'[^A-Za-z0-9]','',s)) < 5:
        return 'EXPECTED_REFERENCE'
    if field in BOOLEAN_FIELDS:
        if isinstance(value,bool):
            return None
        if s.upper() not in {'TRUE','FALSE','VRAI','FAUX','OUI','NON','1','0'}:
            return 'EXPECTED_BOOLEAN'
    if field in TEXT_FIELDS_REJECT_DATE and looks_like_date(s):
        return 'EXPECTED_TEXT_GOT_DATE'
    return None


def coherence_issues_for_data(doc_type,data):
    """Contrôles prudents : uniquement contradictions évidentes."""
    issues=[]
    pairs={
        'ENGAGEMENT_DOMICILIATION': [('DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','CONTRACT_DATE_ORDER')],
        'CONTRAT_TRAVAIL': [('CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','PERMIT_DATE_ORDER')],
        'CONTRAT_SPECIFIQUE': [('CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','PERMIT_DATE_ORDER')],
        'TITRE_TRAVAIL': [('TTR_DATE_DEBUT','TTR_DATE_FIN','WORK_PERMIT_DATE_ORDER')],
    }
    for f1,f2,code in pairs.get(doc_type,[]):
        d1=parse_date_for_coherence((data or {}).get(f1))
        d2=parse_date_for_coherence((data or {}).get(f2))
        if d1 and d2 and d1 > d2:
            issues.append({'code':code,'fields':[f1,f2],'detail':f'{f1}>{f2}'})

    # Cohérence durée TTR uniquement si les 3 éléments sont clairement interprétables.
    if doc_type=='TITRE_TRAVAIL':
        d1=parse_date_for_coherence((data or {}).get('TTR_DATE_DEBUT'))
        d2=parse_date_for_coherence((data or {}).get('TTR_DATE_FIN'))
        dur=_clean_scalar_text((data or {}).get('TTR_DUREE')).upper()
        m=re.search(r'\b(\d{1,2})\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES)\b',dur)
        if d1 and d2 and m and d2>=d1:
            years=int(m.group(1))
            expected_days=years*365.25
            observed=(d2-d1).days
            # Tolérance large : uniquement incohérences manifestes.
            if years>0 and abs(observed-expected_days) > 120:
                issues.append({'code':'TTR_DURATION_DATE_INCONSISTENT',
                               'fields':['TTR_DUREE','TTR_DATE_DEBUT','TTR_DATE_FIN'],
                               'detail':f'duration={dur};days={observed}'})
    return issues


def assess_extraction_data(doc_type,data,active_fields=None):
    """Contrôles sémantiques/cohérence/remplissage.
    V14 : active_fields restreint l'évaluation (ex. profil TTR7 : 7 champs actifs).
    Le remplissage et les champs critiques ne portent que sur active_fields ;
    les champs inactifs restent à null dans raw_data sans pénaliser le statut."""
    expected=CHAMPS_ATTENDUS.get(doc_type) or []
    eval_fields=list(active_fields) if active_fields else expected
    eval_set=set(eval_fields)
    semantic=[]
    for f in expected:
        issue=semantic_issue_for_field(f,(data or {}).get(f))
        if issue and f in eval_set:
            semantic.append({'field':f,'code':issue,'value':(data or {}).get(f)})
    coherence=coherence_issues_for_data(doc_type,data or {})
    critical_missing=[]
    for f in sorted(set(CRITICAL_FIELDS.get(doc_type,set())) & eval_set):
        v=(data or {}).get(f)
        if is_missing_raw(v):
            critical_missing.append(f)
    fill=taux_remplissage(data or {},eval_fields)
    trigger=set(critical_missing)
    trigger.update(x['field'] for x in semantic)
    for x in coherence:
        trigger.update(x.get('fields') or [])
    trigger &= eval_set
    low_fill=fill<SEUIL_REMPLISSAGE_MIN
    return {
        'critical_missing':critical_missing,
        'semantic_issues':semantic,
        'coherence_issues':coherence,
        'trigger_fields':sorted(trigger),
        'fill_rate':fill,
        'low_fill':bool(low_fill),
        'has_problem':bool(trigger or low_fill),
        'evaluated_fields':eval_fields,
    }


def build_page_recovery_prompt(doc_type, issue_report, pass_no):
    problems=[]
    for f in issue_report.get('critical_missing') or []:
        problems.append(f'{f}: MISSING')
    for x in issue_report.get('semantic_issues') or []:
        problems.append(f"{x['field']}: {x['code']} (lu={x.get('value')!r})")
    for x in issue_report.get('coherence_issues') or []:
        problems.append(f"{x['code']}: {','.join(x.get('fields') or [])}")
    if issue_report.get('low_fill'):
        problems.append(f"LOW_FILL_RATE={issue_report.get('fill_rate')}")
    problem_text='\n'.join('- '+p for p in problems) or '- confirmation de la lecture précédente'

    return f"""
RECOVERY PAGE ENTIÈRE — PASS {pass_no}

Relis TOUTE la page de type {doc_type}. Ne relis aucune autre page du PDF.
La lecture précédente a produit au moins une donnée manquante, sémantiquement
incompatible ou incohérente, ou une correction doit être confirmée.

IMPORTANT LAYOUT : sur ces formulaires, la couche des valeurs peut être décalée
verticalement de façon globale VERS LE HAUT ou VERS LE BAS. Le même décalage
peut affecter tous les champs de la page. N'associe donc jamais une valeur à un
libellé uniquement par alignement horizontal. Utilise l'ordre du formulaire,
les champs voisins et le TYPE de valeur attendu.

Exemples de contradictions qui doivent être corrigées par relecture :
- un champ DATE contenant ORAN, INDE ou un poste ;
- un champ LIEU contenant une date ;
- date de début postérieure à date de fin ;
- valeurs décalées d'une ligne à cause de l'impression.

SI LE TYPE EST TITRE_TRAVAIL :
- TTR_DUREE doit être une vraie durée avec unité temporelle, par exemple
  « 2 ANS, 0 JOURS », « 1 AN » ou « 24 MOIS » ;
- une continuation du poste comme « TAGE / LAMINOR 2 » n'est PAS une durée ;
- si TTR_DUREE reste illisible, retourne null pour ce champ mais ne supprime
  jamais pour cette raison des dates début/fin correctement lues ;
- repère la zone comprise entre la valeur de « Durée » et la valeur de
  « Lieu de travail » ;
- les deux dates de validité présentes dans cette zone doivent être lues
  DE HAUT EN BAS ;
- première date = TTR_DATE_DEBUT ;
- seconde date = TTR_DATE_FIN ;
- ne te fie pas à l'alignement horizontal avec « Du » et « Au » ;
- confirme seulement que début < fin ;
- ne contrôle pas ici la compatibilité avec TTR_DUREE : ce contrôle appartient à la Partie 2 ;
- si cette structure reste ambiguë, retourne null plutôt que d'inventer.

PROBLÈMES / POINTS À CONFIRMER :
{problem_text}

Tu dois néanmoins réextraire TOUS les champs de cette page afin de disposer du
contexte complet. Ne permute jamais automatiquement des valeurs. Si une valeur
reste ambiguë, retourne null.

{PROMPTS_EXTRACTION[doc_type]}
"""

print('✅ Prompts V13 chargés — 99 champs inchangés + contrôles sémantiques/layout')
print('   Types gérés          :', ', '.join(sorted(PROMPTS_EXTRACTION)))
print('   Champs TITRE_TRAVAIL :', len(CHAMPS_ATTENDUS['TITRE_TRAVAIL']))
print('   Champs critiques TITRE:', len(CRITICAL_FIELDS['TITRE_TRAVAIL']))


## 10. Vérification du contrat de schéma

In [ ]:
FIELD_SCHEMA = {k:list(v) for k,v in CHAMPS_ATTENDUS.items()}
_current_hash = hashlib.sha256(
    json.dumps(FIELD_SCHEMA, sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()
assert _current_hash == FIELD_SCHEMA_HASH, (_current_hash, FIELD_SCHEMA_HASH)
assert sum(len(v) for v in FIELD_SCHEMA.values()) == 99
print('✅ Schéma exact vérifié : 99 champs | hash', FIELD_SCHEMA_HASH[:16]+'…')
for k,v in FIELD_SCHEMA.items():
    print(f'   {k:28} : {len(v):2d} champs')


## 11. Checkpoints RAW

In [ ]:
def canonical_checkpoint_path(pdf_path):
    return JSON_DIR / f'{pdf_path.stem}.json'


def checkpoint_is_complete(dossier,pdf_path):
    if not isinstance(dossier,dict):
        return False
    if dossier.get('schema_version') != SCHEMA_VERSION:
        return False
    if dossier.get('pipeline_version') != PIPELINE_VERSION:
        return False
    if dossier.get('field_schema_hash') != FIELD_SCHEMA_HASH:
        return False
    if dossier.get('source_file') != pdf_path.name:
        return False
    if not dossier.get('page_records'):
        return False
    try:
        return dossier.get('source_sha256') == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    p=canonical_checkpoint_path(pdf_path)
    if not (RESUME and p.exists()):
        return None
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        return None
    return d if checkpoint_is_complete(d,pdf_path) else None


## 12. TTR-7 — prompt structural + validateur

V14 : `TTR_NUMERO_PERMIS` est la **structure complète** `NN-NNNNNNNN / NN-NN-NNNNNN`
(le « / » fait partie de la référence). Le validateur `_ttr7_valid` exige cette
forme complète ; une référence partielle déclenche le recovery 2400.


In [ ]:

# =====================================================================
# V13.8 — TTR-7 FAST STRUCTURAL
# =====================================================================
TTR7_ACTIVE_FIELDS=(
 "TTR_NUMERO_PERMIS","TTR_NOM","TTR_PRENOM","TTR_DATE_NAISSANCE",
 "TTR_NATIONALITE","TTR_DATE_DEBUT","TTR_DATE_FIN",
)
TTR7_INACTIVE_FIELDS=tuple(
 f for f in CHAMPS_ATTENDUS["TITRE_TRAVAIL"] if f not in TTR7_ACTIVE_FIELDS
)
TTR7_PROFILE="TTR_7_FAST_STRUCTURAL_V1"

PROMPT_TTR7_FAST_STRUCTURAL=r"""
Tu lis UNE page TITRE DE TRAVAIL / PERMIS DE TRAVAIL.

OBJECTIF:
extraire UNIQUEMENT ces 7 champs:
TTR_NUMERO_PERMIS, TTR_NOM, TTR_PRENOM, TTR_DATE_NAISSANCE,
TTR_NATIONALITE, TTR_DATE_DEBUT, TTR_DATE_FIN.

REGLES GENERALES:
- Lis uniquement ce qui est visible.
- N'invente jamais une valeur.
- Ne complete jamais un chiffre ou une date.
- N'utilise jamais une autre suite de chiffres de la page pour remplacer une valeur illisible.
- Ne corrige jamais automatiquement O/0, I/1, S/5, B/8, etc.
- Si un champ reste ambigu apres lecture attentive, retourne null.

TTR_NUMERO_PERMIS — REGLE STRUCTURELLE PRIORITAIRE:
1. Cherche d'abord la ZONE D'EN-TETE du titre/permis, avant le bloc "Le titulaire du present permis..." / avant la description du poste.
2. Le numero de permis est la reference administrative imprimee dans cette zone d'en-tete.
3. Sur les formulaires observes, cette reference est UNE STRUCTURE COMPLETE de la forme:
   NN-NNNNNNNN / NN-NN-NNNNNN
   Exemple de STRUCTURE uniquement (chiffres fictifs):
   23-00009138 / 31-25-001868
4. Le "/" fait partie de la reference : transcris TOUJOURS la structure complete,
   avec ses deux parties et le "/". Ne retourne JAMAIS une seule des deux parties.
5. IMPORTANT: une petite suite numerique isolee ailleurs sur la page
   (tampon, manuscrit, code, date, identifiant, numero partiel, etc.)
   N'EST PAS le numero de permis.
6. Ne choisis JAMAIS un candidat uniquement parce qu'il est numerique.
   Il doit etre dans la zone d'en-tete ET avoir la structure complete "A / B".
7. Si la structure complete n'est pas lisible avec certitude, retourne null
   plutot qu'une reference partielle.
8. Conserve les tirets et le "/" significatifs. Les espaces typographiques
   ne sont pas importants.

ATTENTION AU DECALAGE:
les valeurs peuvent etre decalees verticalement ensemble vers le haut ou le bas.
Le POSTE peut prendre UNE ou DEUX lignes. Ne fais jamais une association par
alignement horizontal fixe.

Pour les dates, utilise la STRUCTURE RELATIVE:
POSTE (1 ou 2 lignes) -> DUREE -> DU/premiere date -> AU/deuxieme date
-> LIEU DE TRAVAIL.
POSTE, DUREE et LIEU sont seulement des ANCRES.
Premiere date apres DUREE = TTR_DATE_DEBUT.
Deuxieme date = TTR_DATE_FIN.
Si ambigu, retourne null. Ne devine pas.

Retourne STRICTEMENT:
{
 "TTR_NUMERO_PERMIS": null,
 "TTR_NOM": null,
 "TTR_PRENOM": null,
 "TTR_DATE_NAISSANCE": null,
 "TTR_NATIONALITE": null,
 "TTR_DATE_DEBUT": null,
 "TTR_DATE_FIN": null,
 "_TTR7_EVIDENCE": {
   "ancre_duree_trouvee": false,
   "ancre_lieu_travail_trouvee": false,
   "poste_lignes": null,
   "date_1_observee": null,
   "date_2_observee": null,
   "duration_text_observe": null,
   "structure_dates_claire": false,
   "permit_zone_entete_trouvee": false,
   "permit_reference_complete_observee": null,
   "permit_candidat_1": null,
   "permit_candidat_2": null,
   "permit_choix_justifie": false
 }
}
"""

def _ttr7_full_raw(parsed):
    parsed=parsed if isinstance(parsed,dict) else {}
    raw={f:None for f in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]}
    for f in TTR7_ACTIVE_FIELDS: raw[f]=parsed.get(f)
    return raw

def _ttr7_evidence(parsed):
    e=parsed.get("_TTR7_EVIDENCE",{}) if isinstance(parsed,dict) else {}
    return e if isinstance(e,dict) else {}

def _ttr7_missing(raw):
    return [f for f in TTR7_ACTIVE_FIELDS if raw.get(f) in (None,"",[])]

def _ttr7_dates_ok(raw):
    a=parse_date_for_coherence(raw.get("TTR_DATE_DEBUT"))
    b=parse_date_for_coherence(raw.get("TTR_DATE_FIN"))
    return bool(a and b and a<b)

def _ttr7_valid(r):
    raw=(r.get('raw_data') or {}) if isinstance(r,dict) else {}
    _permit_compact=re.sub(r'\s+','',str(raw.get('TTR_NUMERO_PERMIS') or '').strip())
    if not re.fullmatch(PERMIT_FULL_RE,_permit_compact):
        return False
    ev=(r.get('ttr7_evidence') or {}) if isinstance(r,dict) else {}
    if ev and (ev.get('permit_zone_entete_trouvee') is False or ev.get('permit_choix_justifie') is False):
        return False
    if _ttr7_missing(raw) or not _ttr7_dates_ok(raw): return False
    if not ev.get("structure_dates_claire"): return False
    if not ev.get("ancre_duree_trouvee") or not ev.get("ancre_lieu_travail_trouvee"): return False
    dur=ev.get("duration_text_observe")
    if dur and not _pair_duration_compatible(raw.get("TTR_DATE_DEBUT"),raw.get("TTR_DATE_FIN"),dur):
        return False
    return True

# V14 : le numéro de permis TTR est la structure COMPLÈTE "NN-NNNNNNNN / NN-NN-NNNNNN".
PERMIT_FULL_RE = r'\d{2}-\d{8}/\d{2}-\d{2}-\d{6}'


In [ ]:
# V14 — non-régression TTR : structure complète obligatoire
assert re.fullmatch(PERMIT_FULL_RE, re.sub(r'\s+','',"23-00009138 / 31-25-001868"))
assert re.fullmatch(PERMIT_FULL_RE, "21-00002974/31-25-001448")
assert not re.fullmatch(PERMIT_FULL_RE, "23-00009138")   # une seule partie = invalide
assert not re.fullmatch(PERMIT_FULL_RE, "0673417")
print("✅ V14 : référence permis = structure complète 'NN-NNNNNNNN / NN-NN-NNNNNN'")


## 13. Confiance opérationnelle & finalisation

In [ ]:
def _confidence_band(score):
    if score>=CONFIDENCE_HIGH: return 'HIGH'
    if score>=CONFIDENCE_MEDIUM: return 'MEDIUM'
    return 'LOW'


def _compute_field_confidence(record,field,final_report):
    value=(record.get('raw_data') or {}).get(field)
    if is_missing_raw(value):
        return {'score':0,'band':'LOW','basis':'MISSING','attempts_supporting':0,'distinct_valid_values':0}
    sem=semantic_issue_for_field(field,value)
    final_issue_fields=set(final_report.get('trigger_fields') or [])
    candidates=(record.get('field_candidates') or {}).get(field,[])
    valid=[c for c in candidates if c.get('semantic_valid')]
    key=_candidate_key(value)
    supporting=sum(1 for c in valid if _candidate_key(c.get('value'))==key)
    distinct=len({_candidate_key(c.get('value')) for c in valid if _candidate_key(c.get('value')) is not None})
    revised=any(x.get('field')==field for x in (record.get('field_revisions') or []))
    disagreement=any(x.get('field')==field for x in (record.get('field_disagreements') or []))

    if sem is not None or field in final_issue_fields:
        score=25; basis='UNRESOLVED_SEMANTIC_OR_COHERENCE'
    elif supporting>=2 and distinct==1:
        score=97 if revised else 95; basis='CONSENSUS_2_PLUS'
    elif supporting>=2 and distinct>1:
        score=82; basis='CONSENSUS_WITH_OTHER_DISAGREEMENT'
    elif disagreement or distinct>1:
        score=58; basis='VALID_VALUES_DISAGREE'
    elif revised:
        score=82; basis='RECOVERY_CORRECTED_SINGLE_SUPPORT'
    elif record.get('page_recovery_triggered'):
        score=84; basis='VALID_AFTER_PAGE_RECOVERY'
    else:
        score=88; basis='VALID_SINGLE_PASS'
    return {
        'score':score,'band':_confidence_band(score),'basis':basis,
        'attempts_supporting':supporting,'distinct_valid_values':distinct,
        'semantic_valid':sem is None,
    }


def _duration_months_approx(text):
    if not text:
        return None
    t=str(text).upper().replace(',', ' ')
    years=re.search(r'\b(\d{1,3})\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES)\b',t)
    months=re.search(r'\b(\d{1,3})\s*MOIS\b',t)
    days=re.search(r'\b(\d{1,4})\s*(?:JOUR|JOURS)\b',t)
    total=0.0; found=False
    if years: total+=12*int(years.group(1)); found=True
    if months: total+=int(months.group(1)); found=True
    if days: total+=int(days.group(1))/30.44; found=True
    return total if found else None

def _pair_duration_compatible(start_value,end_value,duration_text):
    """Contrôle local de plausibilité ; ne modifie jamais le RAW.
    Une durée illisible n'invalide pas des dates valides."""
    d1,d2=parse_date_for_coherence(start_value),parse_date_for_coherence(end_value)
    if not (d1 and d2 and d1<d2):
        return False
    dm=_duration_months_approx(duration_text)
    if dm is None:
        return True
    observed=(d2-d1).days/30.44
    return abs(observed-dm) <= 1.5


def finalize_extraction_record(record):
    dt=record.get('doc_type')
    if dt not in PROMPTS_EXTRACTION:
        record['extraction_status']='NON_APPLICABLE'
        record['extraction_confidence_score']=0.0
        record['extraction_confidence_band']='LOW'
        return record
    expected=CHAMPS_ATTENDUS.get(dt) or []
    for f in expected:
        record['raw_data'].setdefault(f,None)

    # V14 : pour le profil TTR7, l'évaluation (remplissage, champs critiques,
    # statut, confiance) porte sur les 7 champs actifs. Les 14 champs inactifs
    # restent présents à null dans raw_data (schéma inchangé) sans pénaliser
    # le statut de la page.
    is_ttr7 = (dt=='TITRE_TRAVAIL' and record.get('ttr_extraction_profile')==TTR7_PROFILE)
    eval_fields = list(TTR7_ACTIVE_FIELDS) if is_ttr7 else expected
    record['evaluated_fields']=eval_fields
    record['unevaluated_fields']=[f for f in expected if f not in set(eval_fields)]

    record['extraction_taux_remplissage']=taux_remplissage(record['raw_data'],eval_fields)
    final_report=assess_extraction_data(dt,record['raw_data'],active_fields=eval_fields)
    record['critical_fields_missing']=final_report.get('critical_missing') or []
    record['semantic_issues_final']=final_report.get('semantic_issues') or []
    record['coherence_issues_final']=final_report.get('coherence_issues') or []

    flags=list(record.get('quality_flags') or [])
    if record['critical_fields_missing']: flags.append('CRITICAL_FIELD_MISSING_FINAL')
    if record['semantic_issues_final']: flags.append('SEMANTIC_FIELD_MISMATCH_FINAL')
    if record['coherence_issues_final']: flags.append('CROSS_FIELD_COHERENCE_ERROR_FINAL')
    if record['extraction_taux_remplissage']<SEUIL_REMPLISSAGE_MIN: flags.append('LOW_FILL_RATE_FINAL')
    if record.get('page_recovery_triggered'): flags.append('PAGE_RECOVERY_TRIGGERED')
    if record.get('field_revisions'): flags.append('FIELD_REREAD_BY_PAGE_RECOVERY')
    critical_set=set(CRITICAL_FIELDS.get(dt,set())) & set(eval_fields)
    blocking_disagreements=[d for d in (record.get('field_disagreements') or []) if d.get('field') in critical_set]
    nonblocking_disagreements=[d for d in (record.get('field_disagreements') or []) if d.get('field') not in critical_set]
    record['blocking_field_disagreements']=blocking_disagreements
    if blocking_disagreements: flags.append('RECOVERY_CRITICAL_FIELD_DISAGREEMENT')
    if nonblocking_disagreements: flags.append('RECOVERY_NONCRITICAL_DISAGREEMENT_AUDIT')
    record['quality_flags']=list(dict.fromkeys(flags))

    field_conf={f:_compute_field_confidence(record,f,final_report) for f in eval_fields}
    record['field_confidence']=field_conf
    critical=sorted(critical_set) or eval_fields
    critical_scores=[field_conf[f]['score'] for f in critical if f in field_conf]
    base=sum(critical_scores)/len(critical_scores) if critical_scores else 0.0
    class_conf=max(0.0,min(1.0,float(record.get('classification_confidence',0) or 0))) * 100
    page_score=round(0.90*base + 0.10*class_conf,1)
    if blocking_disagreements:
        page_score=min(page_score,74.0)
    if final_report.get('has_problem'):
        page_score=min(page_score,69.0)
    record['extraction_confidence_score']=page_score
    record['extraction_confidence_band']=_confidence_band(page_score)
    record['confidence_method']=CONFIDENCE_METHOD

    any_value=any(not is_missing_raw(record['raw_data'].get(f)) for f in eval_fields)
    if not any_value:
        record['extraction_status']='JSON_VIDE'
    elif final_report.get('has_problem') or blocking_disagreements:
        record['extraction_status']='PARTIELLE'
    else:
        record['extraction_status']='OK'
    return record

print('✅ V14 : confiance/finalisation sur champs actifs (profil TTR7)')


# Préflight runtime

Vérification, avant chaque PDF, que toutes les fonctions et constantes critiques du pipeline V14 sont bien chargées.

In [ ]:
# =====================================================================
# V13.8.7 — PREFLIGHT RUNTIME RENFORCE
# =====================================================================
# Important : cette cellule DEFINIT le contrôle. Le contrôle est exécuté
# au début de process_pdf(), après chargement de toutes les fonctions.

def runtime_preflight_v14():
    required_functions = [
        "parse_date_for_coherence", "_pair_duration_compatible",
        "_candidate_key", "_add_field_candidate",
        "finalize_extraction_record", "extract_classified_pages",
        "build_recovery_jobs", "run_extraction_jobs", "_merge_job_result",
        "_page_recovery_image", "_render_for_strategy",
        "compact_prompt_for_doc", "assess_extraction_data",
        "resolve_orientations", "detect_quarter_turn", "deskew_for_vlm", "rotate_pil_expand_white",
        "_rotate_clockwise", "_parse_orientation",
    ]
    required_constants = [
        "PROMPTS_EXTRACTION", "CHAMPS_ATTENDUS",
        "IMAGE_MAX_SIZE", "IMAGE_MAX_SIZE_HAUTE_DEF",
        "IMAGE_MAX_SIZE_RECOVERY_2",
        "GPU_BATCH_SIZE_EXTRACTION_STANDARD", "GPU_BATCH_SIZE_EXTRACTION_HD",
        "TTR7_ACTIVE_FIELDS", "TTR7_PROFILE", "PROMPT_TTR7_FAST_STRUCTURAL",
    ]
    missing_f = [x for x in required_functions
                 if x not in globals() or not callable(globals()[x])]
    missing_c = [x for x in required_constants if x not in globals()]
    if missing_f or missing_c:
        msg=[]
        if missing_f: msg.append("fonctions=" + ", ".join(missing_f))
        if missing_c: msg.append("constantes=" + ", ".join(missing_c))
        raise RuntimeError("PREFLIGHT V14 ECHEC - " + " | ".join(msg))
    print("✅ PREFLIGHT V14 : dépendances runtime OK")
    print("✅ ORIENTATION : résolution VLM (toutes pages extractibles) + deskew avant extraction")
    print("✅ PTR : aucun appel Qwen d'extraction")
    print("✅ STANDARD : DOM/CTR/CTS conservés en batch standard")

print("✅ PREFLIGHT V14 chargé — il sera exécuté avant chaque PDF")


# Moteur d'extraction V14 — orientation résolue en amont, recovery page entière 2400px

In [ ]:
def _new_record_from_page(page, parsed, output):
    doc_type=parsed.get('type_document') or parsed.get('type') or 'AUTRE'
    try:
        confidence=float(parsed.get('confidence',0) or 0)
    except Exception:
        confidence=0.0
    bloc_identite=bool(parsed.get('bloc_identite_present'))
    if doc_type not in TYPES_VALIDES:
        doc_type='AUTRE'
    requalifie=False
    if bloc_identite and doc_type in ('PERMIS_TRAVAIL_COUVERTURE','AUTRE'):
        doc_type='TITRE_TRAVAIL'; confidence=max(confidence,CLASSIFICATION_THRESHOLD); requalifie=True
    if confidence<CLASSIFICATION_THRESHOLD and not requalifie:
        doc_type='AUTRE'
    return {
        'page_num':page['page_num'],'width':page['width'],'height':page['height'],
        'white_ratio':page['white_ratio'],'image':page['image'],
        'doc_type':doc_type,'titre_detecte':parsed.get('titre_detecte'),
        'bloc_identite_present':bloc_identite,'classification_requalifiee':requalifie,
        'classification_retry_fullres':False,'classification_confidence':confidence,
        'classification_raw_text':output.get('text'),
        'classification_attempts':[{
            'strategy':'LOWRES_1100','raw_text':output.get('text'),
            'parsed':parsed,'tokens_in':output.get('tokens_in',0),
            'tokens_out':output.get('tokens_out',0),'elapsed_s':output.get('elapsed_s',0),
        }],
        'classification_tokens_in':int(output.get('tokens_in',0) or 0),
        'classification_tokens_out':int(output.get('tokens_out',0) or 0),
        'classification_elapsed_s':float(output.get('elapsed_s',0) or 0),
        'raw_data':{},'extraction_status':'NON_LANCEE','extraction_error':None,
        'extraction_raw_text':None,'extraction_attempts':[],
        'extraction_strategies':[],'extraction_taux_remplissage':0.0,
        'extraction_call_count':0,'retry_call_count':0,
        'extraction_tokens_in':0,'extraction_tokens_out':0,'extraction_elapsed_s':0.0,
        'field_revisions':[],'critical_fields_missing':[],'quality_flags':[],
        'field_candidates':{},'field_disagreements':[],'recovery_history':[],
        'semantic_issues_initial':[],'semantic_issues_final':[],'coherence_issues_final':[],
        'page_recovery_triggered':False,'recovery_passes':0,
        'field_confidence':{},'extraction_confidence_score':0.0,
        'extraction_confidence_band':'LOW','confidence_method':CONFIDENCE_METHOD,
        'is_virtual_subdocument':False,
        # V14 — géométrie de page, décidée AVANT extraction par resolve_orientations()
        'rotation_clockwise':0,'rotation_applied':False,
        'orientation_quarter_detect':None,'orientation_quarter_margin':None,
        'orientation_raw_text':None,'orientation_tokens_in':0,
        'orientation_tokens_out':0,'orientation_elapsed_s':0.0,
        'deskew_enabled':bool(DESKEW_ENABLED),'deskew_applied':False,
        'deskew_detected_angle_deg':None,'deskew_correction_angle_deg':0.0,
        'deskew_evidence_lines':0,
    }


def classify_pages(pages):
    records=[]
    for start in range(0,len(pages),GPU_BATCH_SIZE_CLASSIFICATION):
        batch=pages[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
        outs=ask_batch(PROMPT_CLASSIFICATION,[image_for_classification(x['image']) for x in batch],MAX_NEW_TOKENS_CLASSIFICATION)
        _shared_cls=sum(float(o.get('elapsed_s',0) or 0) for o in outs)
        for page,out in zip(batch,outs):
            _rec=_new_record_from_page(page,parse_json_response(out['text']),out)
            records.append(_rec)
            page_pevent(
                page.get('page_num'), _rec.get('doc_type'),
                'CLASSIFICATION','CLASSIFICATION_BATCH',
                _shared_cls,len(batch),
                out.get('tokens_in',0),out.get('tokens_out',0)
            )

    if CLASSIFICATION_RETRY_ON_AUTRE or CLASSIFICATION_RETRY_LOW_CONFIDENCE:
        retry=[r for r in records if (r['doc_type']=='AUTRE' or float(r.get('classification_confidence',0) or 0) < CLASSIFICATION_HARD_MIN_CONFIDENCE)]
        for start in range(0,len(retry),GPU_BATCH_SIZE_CLASSIFICATION):
            batch=retry[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
            outs=ask_batch(PROMPT_CLASSIFICATION,[r['image'] for r in batch],MAX_NEW_TOKENS_CLASSIFICATION)
            _shared_cls_retry=sum(float(o.get('elapsed_s',0) or 0) for o in outs)
            for r,out in zip(batch,outs):
                parsed=parse_json_response(out['text'])
                page_pevent(
                    r.get('page_num'), parsed.get('type_document') or parsed.get('type') or r.get('doc_type'),
                    'CLASSIFICATION_RETRY','FULLRES_1400_RETRY',
                    _shared_cls_retry,len(batch),
                    out.get('tokens_in',0),out.get('tokens_out',0)
                )
                attempt={'strategy':'FULLRES_1400_RETRY','raw_text':out.get('text'),'parsed':parsed,
                         'tokens_in':out.get('tokens_in',0),'tokens_out':out.get('tokens_out',0),'elapsed_s':out.get('elapsed_s',0)}
                new=_new_record_from_page({'page_num':r['page_num'],'width':r['width'],'height':r['height'],
                                           'white_ratio':r['white_ratio'],'image':r['image']},parsed,out)
                attempts=list(r.get('classification_attempts') or [])+[attempt]
                new['classification_attempts']=attempts
                new['classification_tokens_in']=sum(int(a.get('tokens_in',0) or 0) for a in attempts)
                new['classification_tokens_out']=sum(int(a.get('tokens_out',0) or 0) for a in attempts)
                new['classification_elapsed_s']=round(sum(float(a.get('elapsed_s',0) or 0) for a in attempts),3)
                new['classification_raw_text']='\n\n'.join(f"[{a['strategy']}] {a.get('raw_text','')}" for a in attempts)
                new['classification_retry_fullres']=True
                r.clear(); r.update(new)
    return records



def apply_regulatory_classification_gate(records):
    """Bloque l'extraction si la classification n'est pas suffisamment sûre.

    Important : ce garde-fou ne prétend pas garantir 100 % de justesse. Il empêche
    surtout qu'un schéma d'extraction soit appliqué à une page classée AUTRE ou à
    une classification restant sous le seuil après éventuel retry pleine résolution.
    """
    for r in records:
        flags=list(r.get('quality_flags') or [])
        conf=float(r.get('classification_confidence',0) or 0)
        dt=r.get('doc_type')
        review = (dt not in PROMPTS_EXTRACTION) or (conf < CLASSIFICATION_HARD_MIN_CONFIDENCE)
        r['classification_review_required']=bool(review)
        if review:
            flags.append('CLASSIFICATION_REVIEW_REQUIRED')
        r['quality_flags']=list(dict.fromkeys(flags))
    return records


def add_virtual_permit_subdocuments(records):
    """V13.7 : aucune lecture Qwen de la couverture permis.

    Compatibilité : les trois champs PTR restent représentés par un enregistrement
    désactivé avec valeurs nulles. Aucun crop, aucun token, aucun recovery.
    """
    ptr_fields = list(CHAMPS_ATTENDUS.get('PERMIS_TRAVAIL_COUVERTURE') or [])
    physical_cover = [
        r for r in records
        if r.get('doc_type') == 'PERMIS_TRAVAIL_COUVERTURE'
    ]

    # Si une page physique a été classée couverture, on la conserve pour l'audit
    # mais son extraction est explicitement désactivée.
    for r in physical_cover:
        r['raw_data'] = {f: None for f in ptr_fields}
        r['extraction_status'] = 'DISABLED_V13_7'
        r['extraction_error'] = None
        r['extraction_call_count'] = 0
        r['retry_call_count'] = 0
        r['page_recovery_triggered'] = False
        r['recovery_passes'] = 0
        r['extraction_strategies'] = []
        r['extraction_attempts'] = []
        r['quality_flags'] = list(dict.fromkeys(
            list(r.get('quality_flags') or []) + ['PTR_EXTRACTION_DISABLED_V13_7']
        ))

    if physical_cover:
        return records

    # S'il n'existe pas de page physique dédiée, créer uniquement une trace
    # synthétique rattachée à la page TTR. Elle ne déclenche aucun appel Qwen.
    ttr = next((r for r in records if r.get('doc_type') == 'TITRE_TRAVAIL'), None)
    if ttr is None:
        return records

    v = {
        'page_num': ttr.get('page_num'),
        'width': ttr.get('width'),
        'height': ttr.get('height'),
        'white_ratio': ttr.get('white_ratio'),
        'image': ttr.get('image'),
        'doc_type': 'PERMIS_TRAVAIL_COUVERTURE',
        'titre_detecte': 'EXTRACTION_DESACTIVEE_V13_7',
        'bloc_identite_present': False,
        'classification_requalifiee': False,
        'classification_retry_fullres': False,
        'classification_confidence': ttr.get('classification_confidence',0),
        'classification_raw_text': 'AUCUN_APPEL_QWEN_PTR_V13_7',
        'classification_attempts': [],
        'classification_tokens_in': 0,
        'classification_tokens_out': 0,
        'classification_elapsed_s': 0.0,
        'raw_data': {f: None for f in ptr_fields},
        'extraction_status': 'DISABLED_V13_7',
        'extraction_error': None,
        'extraction_raw_text': None,
        'extraction_attempts': [],
        'extraction_strategies': [],
        'extraction_taux_remplissage': 0.0,
        'extraction_tokens_in': 0,
        'extraction_tokens_out': 0,
        'extraction_call_count': 0,
        'retry_call_count': 0,
        'extraction_elapsed_s': 0.0,
        'field_revisions': [],
        'critical_fields_missing': [],
        'field_candidates': {},
        'field_disagreements': [],
        'recovery_history': [],
        'semantic_issues_initial': [],
        'semantic_issues_final': [],
        'coherence_issues_final': [],
        'page_recovery_triggered': False,
        'recovery_passes': 0,
        'field_confidence': {},
        'extraction_confidence_score': None,
        'extraction_confidence_band': 'DISABLED',
        'confidence_method': CONFIDENCE_METHOD,
        'quality_flags': ['PTR_EXTRACTION_DISABLED_V13_7'],
        'is_virtual_subdocument': True,
        'virtual_parent_doc_type': 'TITRE_TRAVAIL',
        'ptr_extraction_disabled': True,
    }
    return records + [v]


def _render_for_strategy(record,pdf_path,strategy_name,crop=None,max_side=None):
    if strategy_name=='STANDARD':
        # Image déjà orientée + deskewée en mémoire par resolve_orientations().
        return record['image']
    img=render_page_region(pdf_path,record['page_num']-1,zoom=PDF_ZOOM_HAUTE_DEF,
                           max_side=max_side or IMAGE_MAX_SIZE_HAUTE_DEF,crop=crop)
    # V14 : tout re-rendu HD repart du PDF BRUT. Il faut donc REJOUER la géométrie
    # décidée en phase ORIENTATION : rotation quart de tour puis deskew, dans cet ordre.
    img=_rotate_clockwise(img,int(record.get('rotation_clockwise',0) or 0))
    if record.get('deskew_applied'):
        _corr=float(record.get('deskew_correction_angle_deg',0) or 0)
        if abs(_corr)>0:
            img=rotate_pil_expand_white(img,_corr)
    return img



def _initial_job(r,pdf_path):
    dt=r['doc_type']

    # TTR : page entière en HD 1800, avec le prompt structurel TTR-7.
    # Pas de dépendance à DOCS_HD / IMAGE_MAX_SIZE_HD / get_page_image_for_size.
    if dt=='TITRE_TRAVAIL':
        _ttr_img=_render_for_strategy(
            r, pdf_path, 'TTR7_FAST_STRUCTURAL',
            crop=(0,1,0,1), max_side=IMAGE_MAX_SIZE_HAUTE_DEF
        )
        return {
          'record':r,
          'prompt':PROMPT_TTR7_FAST_STRUCTURAL,
          'image':_ttr_img,
          'strategy':'TTR7_FAST_STRUCTURAL',
          'profile':'HD',
          'mode':'INITIAL_TTR7',
          'trigger_fields':[],
          'confirm_fields':[],
          'max_new_tokens':700,
        }

    # DOM / CTR / CTS : conserver le chemin STANDARD historique et le batching.
    # La couverture PTR est exclue avant cette fonction.
    prompt=compact_prompt_for_doc(dt,PROMPTS_EXTRACTION[dt])
    return {
      'record':r,
      'prompt':prompt,
      'image':r['image'],
      'strategy':'STANDARD',
      'profile':'STANDARD',
      'mode':'INITIAL',
      'trigger_fields':[],
      'confirm_fields':[],
      'max_new_tokens':int(MAX_NEW_TOKENS_BY_DOC.get(dt,MAX_NEW_TOKENS_EXTRACTION)),
    }


def _candidate_key(value):
    if value is None: return None
    if isinstance(value,bool): return str(value).lower()
    return re.sub(r'\s+',' ',str(value).strip()).casefold()


def _add_field_candidate(record,field,value,strategy):
    if is_missing_raw(value):
        return
    issue=semantic_issue_for_field(field,value)
    record.setdefault('field_candidates',{}).setdefault(field,[]).append({
        'value':value,
        'strategy':strategy,
        'semantic_valid':issue is None,
        'semantic_issue':issue,
    })


def _merge_job_result(job,output):
    record=job['record']; dt=record['doc_type']; mode=job.get('mode','INITIAL')
    parsed=parse_json_response(output.get('text',''))
    if dt=='TITRE_TRAVAIL' and mode in ('INITIAL_TTR7','RECOVERY_TTR7_2400'):
        record['ttr_extraction_profile']=TTR7_PROFILE
        record['ttr_active_fields']=list(TTR7_ACTIVE_FIELDS)
        record['ttr_inactive_fields_not_extracted']=list(TTR7_INACTIVE_FIELDS)
        record['ttr7_evidence']=_ttr7_evidence(parsed)
        parsed=_ttr7_full_raw(parsed)
    else:
        parsed=expand_compact_extraction(parsed,dt)
    expected=CHAMPS_ATTENDUS.get(dt) or []
    trigger=set(job.get('trigger_fields') or [])
    confirm=set(job.get('confirm_fields') or [])
    added=corrected=agreed=disagreed=0
    changed_fields=[]

    attempt_quality=assess_extraction_data(dt,parsed)
    attempt={
        'strategy':job['strategy'],
        'mode':mode,
        'fields_requested':'ALL_PAGE_FIELDS',
        'fields_returned':sorted([k for k,v in parsed.items() if k in expected and not is_missing_raw(v)]),
        'tokens_in':int(output.get('tokens_in',0) or 0),
        'tokens_out':int(output.get('tokens_out',0) or 0),
        'elapsed_s':float(output.get('elapsed_s',0) or 0),
        'is_retry': mode!='INITIAL',
        'semantic_issue_count':len(attempt_quality.get('semantic_issues') or []),
        'coherence_issue_count':len(attempt_quality.get('coherence_issues') or []),
        'critical_missing_count':len(attempt_quality.get('critical_missing') or []),
        'fill_rate':attempt_quality.get('fill_rate'),
    }
    if STORE_QWEN_RAW_TEXT_IN_ATTEMPTS:
        attempt['raw_text']=output.get('text')
    if STORE_PARSED_DATA_IN_ATTEMPTS:
        attempt['parsed_data']=parsed

    record['extraction_attempts'].append(attempt)
    record['extraction_call_count']=int(record.get('extraction_call_count',0) or 0)+1
    if mode not in ('INITIAL','INITIAL_TTR7'):
        record['retry_call_count']=int(record.get('retry_call_count',0) or 0)+1
        record['recovery_passes']=int(record.get('recovery_passes',0) or 0)+1

    for field in expected:
        if field not in parsed:
            continue
        value=parsed.get(field)
        if is_missing_raw(value):
            continue
        _add_field_candidate(record,field,value,job['strategy'])
        old=record['raw_data'].get(field)
        old_issue=semantic_issue_for_field(field,old) if not is_missing_raw(old) else 'MISSING'
        new_issue=semantic_issue_for_field(field,value)

        if mode in ('INITIAL','INITIAL_TTR7'):
            if is_missing_raw(old):
                record['raw_data'][field]=value; added+=1
            continue

        # Recovery : la page entière est relue mais on ne remplace pas aveuglément
        # tous les champs déjà corrects.
        if is_missing_raw(old):
            if new_issue is None:
                record['raw_data'][field]=value; added+=1; changed_fields.append(field)
            continue

        if _candidate_key(old)==_candidate_key(value):
            agreed+=1
            continue

        # Champ encore objectivement problématique après la lecture précédente :
        # une nouvelle valeur sémantiquement plausible peut le corriger.
        if field in trigger and new_issue is None:
            record['raw_data'][field]=value; corrected+=1; changed_fields.append(field)
            record['field_revisions'].append({
                'field':field,'old':old,'new':value,'strategy':job['strategy'],
                'reason':'PAGE_RECOVERY_TRIGGER_FIELD'
            })
            continue

        # Confirmation d'une correction : si la nouvelle lecture diffère alors que
        # les deux valeurs sont plausibles, on n'écrase pas ; on signale le désaccord.
        if field in confirm:
            disagreed+=1
            record.setdefault('field_disagreements',[]).append({
                'field':field,'kept':old,'candidate':value,'strategy':job['strategy'],
                'reason':'RECOVERY_CONFIRMATION_DISAGREEMENT'
            })
            continue

        # Champ non déclencheur : la page a été relue pour le contexte. Une différence
        # n'autorise pas une correction silencieuse.
        if new_issue is None:
            disagreed+=1
            record.setdefault('field_disagreements',[]).append({
                'field':field,'kept':old,'candidate':value,'strategy':job['strategy'],
                'reason':'NON_TRIGGER_FIELD_DISAGREEMENT'
            })

    attempt['fields_added']=added
    attempt['fields_corrected']=corrected
    attempt['fields_agreed']=agreed
    attempt['fields_disagreed']=disagreed
    attempt['changed_fields']=sorted(set(changed_fields))

    record['extraction_tokens_in']+=int(output.get('tokens_in',0) or 0)
    record['extraction_tokens_out']+=int(output.get('tokens_out',0) or 0)
    record['extraction_elapsed_s']=round(record.get('extraction_elapsed_s',0)+float(output.get('elapsed_s',0) or 0),3)
    record['extraction_taux_remplissage']=taux_remplissage(record['raw_data'],expected)
    record['extraction_strategies'].append({
        'nom':job['strategy'],'mode':mode,'champs_ajoutes':added,
        'champs_corriges':corrected,'accords':agreed,'desaccords':disagreed,
        'taux_apres':record['extraction_taux_remplissage'],
    })
    if mode not in ('INITIAL','INITIAL_TTR7'):
        record.setdefault('recovery_history',[]).append({
            'strategy':job['strategy'],
            'trigger_fields':sorted(trigger),
            'confirm_fields':sorted(confirm),
            'changed_fields':sorted(set(changed_fields)),
            'issue_report_after':assess_extraction_data(dt,record.get('raw_data') or {}),
        })


def _run_job_chunk(chunk):
    if not chunk: return
    max_new=max(int(j.get('max_new_tokens',MAX_NEW_TOKENS_EXTRACTION)) for j in chunk)
    try:
        _psync(); _bt=_pnow()
        outs=ask_batch_mixed([j['prompt'] for j in chunk],[j['image'] for j in chunk],max_new)
        _psync(); _batch_s=_pnow()-_bt
        for j,o in zip(chunk,outs):
            _merge_job_result(j,o)
            _r=j['record']
            page_pevent(
                _r.get('page_num'),_r.get('doc_type'),
                j.get('mode','EXTRACTION'),j.get('strategy'),
                _batch_s,len(chunk),
                o.get('tokens_in',0),o.get('tokens_out',0),
                note='TEMPS_BATCH_PARTAGE' if len(chunk)>1 else 'TEMPS_PAGE_DIRECT'
            )
    except Exception as exc:
        if len(chunk)>1:
            if is_cuda_oom(exc):
                print(f'⚠️ OOM batch {len(chunk)} -> découpage'); gc.collect(); torch.cuda.empty_cache()
            else:
                print(f'⚠️ Batch {len(chunk)} refusé ({type(exc).__name__}) -> sous-batches')
            mid=len(chunk)//2; _run_job_chunk(chunk[:mid]); _run_job_chunk(chunk[mid:]); return
        raise


def run_extraction_jobs(jobs):
    """V13.7 : conserver le batching agressif qui a donné le meilleur temps.
    Les jobs STANDARD sont groupés ensemble et les jobs HD ensemble.
    """
    std = [j for j in jobs if j.get('profile') == 'STANDARD']
    hd  = [j for j in jobs if j.get('profile') != 'STANDARD']

    for k in range(0, len(std), GPU_BATCH_SIZE_EXTRACTION_STANDARD):
        _run_job_chunk(std[k:k+GPU_BATCH_SIZE_EXTRACTION_STANDARD])

    for k in range(0, len(hd), GPU_BATCH_SIZE_EXTRACTION_HD):
        _run_job_chunk(hd[k:k+GPU_BATCH_SIZE_EXTRACTION_HD])


def _page_recovery_image(record,pdf_path,max_side):
    # V14 : relecture de TOUTE LA PAGE depuis le PDF brut ; _render_for_strategy
    # rejoue automatiquement la rotation quart de tour et le deskew décidés
    # en phase ORIENTATION. (La couverture PTR n'est jamais relue : pas d'appel Qwen.)
    return _render_for_strategy(record,pdf_path,'PAGE_RECOVERY',(0,1,0,1),max_side)


def build_recovery_jobs(records,pdf_path,pass_no=None):
    jobs=[]
    for r in records:
        dt=r.get('doc_type')
        if dt=='PERMIS_TRAVAIL_COUVERTURE' or dt not in PROMPTS_EXTRACTION: continue

        if dt=='TITRE_TRAVAIL':
            if _ttr7_valid(r):
                r['ttr7_fast_stop']=True; r['ttr7_recovery_reason']=[]; continue
            raw=r.get('raw_data') or {}; ev=r.get('ttr7_evidence') or {}
            reasons=[]
            miss=_ttr7_missing(raw)
            if miss: reasons.append("MISSING:"+",".join(miss))
            if not _ttr7_dates_ok(raw): reasons.append("DATE_PAIR_INVALID")
            if not ev.get('structure_dates_claire'): reasons.append("DATE_STRUCTURE_NOT_CLEAR")
            if not ev.get('ancre_duree_trouvee'): reasons.append("DURATION_ANCHOR_NOT_FOUND")
            if not ev.get('ancre_lieu_travail_trouvee'): reasons.append("WORKPLACE_ANCHOR_NOT_FOUND")
            r['page_recovery_triggered']=True; r['ttr7_fast_stop']=False
            r['ttr7_recovery_reason']=reasons
            # V14 : on ne déclenche la correction QUE sur les champs réellement
            # défaillants (manquants, paire de dates invalide, permis incomplet,
            # valeur sémantiquement invalide). Les champs valides de la 1re lecture
            # ne peuvent pas être écrasés silencieusement par la relecture.
            trig=set(miss)
            if (not _ttr7_dates_ok(raw)) or (not ev.get('structure_dates_claire')):
                trig.update(('TTR_DATE_DEBUT','TTR_DATE_FIN'))
            _pv=re.sub(r'\s+','',str(raw.get('TTR_NUMERO_PERMIS') or '').strip())
            if not re.fullmatch(PERMIT_FULL_RE,_pv):
                trig.add('TTR_NUMERO_PERMIS')
            for _f in ('TTR_NOM','TTR_PRENOM','TTR_NATIONALITE'):
                if not is_missing_raw(raw.get(_f)) and semantic_issue_for_field(_f,raw.get(_f)):
                    trig.add(_f)
            if not trig:
                trig=set(TTR7_ACTIVE_FIELDS)  # garde-fou : ne jamais relire sans cible
            jobs.append({
              'record':r,
              'prompt':PROMPT_TTR7_FAST_STRUCTURAL+"""
RELECTURE DE SECURITE 2400 PX.
Relis toute la page depuis zero. Ne recopie pas aveuglement la premiere lecture.
""",
              'image':_page_recovery_image(r,pdf_path,IMAGE_MAX_SIZE_RECOVERY_2),
              'strategy':'TTR7_RECOVERY_2400_SINGLE','profile':'HD',
              'mode':'RECOVERY_TTR7_2400','trigger_fields':sorted(trig),
              'confirm_fields':[],'max_new_tokens':700})
            continue

        current=assess_extraction_data(dt,r.get('raw_data') or {})
        r['semantic_issues_initial']=list(current.get('semantic_issues') or [])
        # V14 : RECOVERY_ON_CRITICAL_SEMANTIC_MISMATCH couvre le cas d'une page
        # décalée (valeurs montées/descendues d'une ligne) où les champs critiques
        # sont REMPLIS MAIS INVALIDES (ex. un libellé lu à la place d'une valeur) —
        # cas qui n'activait ni critical_missing ni low_fill en V13.
        _crit=set(CRITICAL_FIELDS.get(dt,set()) or set())
        _crit_sem=[x for x in (current.get('semantic_issues') or []) if x.get('field') in _crit]
        need=bool(ENABLE_PAGE_RECOVERY and (
          (RECOVERY_ON_CRITICAL_MISSING and current.get('critical_missing')) or
          (RECOVERY_ON_COHERENCE_ERROR and current.get('coherence_issues')) or
          (RECOVERY_ON_CRITICAL_SEMANTIC_MISMATCH and _crit_sem) or
          (RECOVERY_ON_LOW_FILL and current.get('low_fill'))))
        if need:
            r['page_recovery_triggered']=True
            jobs.append({'record':r,
              'prompt':compact_prompt_for_doc(dt,build_page_recovery_prompt(dt,current,2400)),
              'image':_page_recovery_image(r,pdf_path,IMAGE_MAX_SIZE_RECOVERY_2),
              'strategy':'PAGE_RECOVERY_2400_SINGLE','profile':'HD',
              'mode':'RECOVERY_2400_SINGLE',
              'trigger_fields':list(current.get('trigger_fields') or []),'confirm_fields':[],
              'max_new_tokens':int(MAX_NEW_TOKENS_RECOVERY_BY_DOC.get(dt,MAX_NEW_TOKENS_RECOVERY))})
    return jobs

def extract_classified_pages(records,pdf_path):
    applicable=[r for r in records
      if r.get('doc_type') in PROMPTS_EXTRACTION
      and r.get('doc_type')!='PERMIS_TRAVAIL_COUVERTURE'
      and not (BLOCK_EXTRACTION_ON_CLASSIFICATION_CONFLICT and r.get('classification_review_required'))]
    # V14 : la géométrie (rotation + deskew) a déjà été résolue pour TOUTES les
    # pages par resolve_orientations() dans process_pdf — rien à refaire ici.
    run_extraction_jobs([_initial_job(r,pdf_path) for r in applicable])
    # V13.8 : aucune observation structurelle Qwen séparée.
    run_extraction_jobs(build_recovery_jobs(applicable,pdf_path))
    for r in records:
        finalize_extraction_record(r)
        if r.get('doc_type')=='TITRE_TRAVAIL':
            r['ttr_extraction_profile']=TTR7_PROFILE
            r['ttr_active_fields']=list(TTR7_ACTIVE_FIELDS)
            r['ttr_inactive_fields_not_extracted']=list(TTR7_INACTIVE_FIELDS)
            r['ttr7_final_valid']=_ttr7_valid(r)
    return records

def print_call_diagnostics(records):
    if not PRINT_CALL_DIAGNOSTICS:
        return
    print('\n--- Diagnostic appels Qwen par page V14 ---')
    for r in records:
        if r.get('doc_type') not in PROMPTS_EXTRACTION:
            continue
        strategies=[a.get('strategy') for a in (r.get('extraction_attempts') or [])]
        print(
            f"page={r.get('page_num')} type={r.get('doc_type')} "
            f"classif={r.get('classification_confidence')} "
            f"calls={r.get('extraction_call_count',0)} retries={r.get('retry_call_count',0)} "
            f"fill={r.get('extraction_taux_remplissage')} "
            f"confidence={r.get('extraction_confidence_score')}({r.get('extraction_confidence_band')}) "
            f"critical={r.get('critical_fields_missing')} "
            f"semantic={len(r.get('semantic_issues_final') or [])} "
            f"coherence={len(r.get('coherence_issues_final') or [])} "
            f"strategies={strategies}"
        )


def _json_safe_record(record):
    return {k:v for k,v in record.items() if k!='image'}


def process_pdf(pdf_path):
    runtime_preflight_v14()
    profiler_reset()
    t0=time.time(); log(f'📁 {pdf_path.name}')
    with pstage("PDF_TO_PAGES",pdf=pdf_path.name):
        pages=pdf_to_pages(pdf_path)
    with pstage("CLASSIFICATION_TOTAL",pages=len(pages)):
        records=classify_pages(pages)
    # V14 : la géométrie est résolue AVANT le gate réglementaire et AVANT toute
    # extraction — rotation quart de tour (VLM, toutes pages extractibles),
    # re-classification unique des pages tournées encore douteuses, puis deskew.
    with pstage("ORIENTATION_TOTAL",pages=len(records)):
        records=resolve_orientations(records)
    records=apply_regulatory_classification_gate(records)
    records=add_virtual_permit_subdocuments(records)
    with pstage("EXTRACTION_TOTAL",pages=len(pages)):
        records=extract_classified_pages(records,pdf_path)
    print_call_diagnostics(records)
    tokens_in=sum(r.get('classification_tokens_in',0)+r.get('orientation_tokens_in',0)+r.get('extraction_tokens_in',0) for r in records)
    tokens_out=sum(r.get('classification_tokens_out',0)+r.get('orientation_tokens_out',0)+r.get('extraction_tokens_out',0) for r in records)
    elapsed=round(time.time()-t0,3)
    physical_records=[r for r in records if not r.get('is_virtual_subdocument')]
    present_doc_types=sorted(set(
        r.get('doc_type') for r in physical_records
        if r.get('doc_type') and r.get('doc_type')!='AUTRE'
    ))
    expected_core=['ENGAGEMENT_DOMICILIATION','CONTRAT_TRAVAIL','CONTRAT_SPECIFIQUE','TITRE_TRAVAIL']
    page_presence={
        'present_doc_types':present_doc_types,
        'missing_core_doc_types':[x for x in expected_core if x not in present_doc_types],
        'classification_review_pages':[
            r.get('page_num') for r in physical_records if r.get('classification_review_required')
        ],
        'physical_pages':len(pages),
        'note':'Absence page != champ OCR vide. La Partie 2 exploite cette distinction.'
    }

    dossier={
        'schema_version':SCHEMA_VERSION,
        'field_schema_hash':FIELD_SCHEMA_HASH,
        'field_schema':FIELD_SCHEMA,
        'source_file':pdf_path.name,
        'source_sha256':sha256_file(pdf_path),
        'pipeline_version':PIPELINE_VERSION,
        'extraction_engine':{
            'model':'Qwen3.6-27B-FP8','model_path':MODEL_PATH,'flash_attn':False,
            'classification_policy':'MANDATORY_VLM_PER_PAGE_NO_PAGE_ORDER',
            'classification_hard_min_confidence':CLASSIFICATION_HARD_MIN_CONFIDENCE,
            'standard_max_side':IMAGE_MAX_SIZE,'classification_max_side':IMAGE_MAX_SIZE_CLASSIFICATION,
            'hd_max_side':IMAGE_MAX_SIZE_HAUTE_DEF,
            'orientation_policy':ORIENTATION_MODE,
            'orientation_max_new_tokens':ORIENTATION_MAX_NEW_TOKENS,
            'deskew_enabled':bool(DESKEW_ENABLED),
            'deskew_range_deg':[DESKEW_MIN_ABS_DEG,DESKEW_MAX_ABS_DEG],
            'recovery_2_max_side':IMAGE_MAX_SIZE_RECOVERY_2,
            'recovery_policy':'V14_SINGLE_WHOLE_PAGE_2400_GEOMETRY_REPLAYED',
            'ptr_extraction_policy':'DISABLED_NO_QWEN_KEEP_3_FIELDS_NULL',
            'confidence_method':CONFIDENCE_METHOD,
            'confidence_note':'Operational indicator; not native Qwen log-probability',
            'created_at':datetime.now().isoformat(timespec='seconds'),
        },
        'stats':{
            'pages':len(pages),'page_records':len(records),
            'virtual_subdocuments':sum(bool(r.get('is_virtual_subdocument')) for r in records),
            'classification_review_required':sum(bool(r.get('classification_review_required')) for r in records),
            'classification_calls':sum(len(r.get('classification_attempts') or []) for r in records),
            'extraction_calls':sum(int(r.get('extraction_call_count',0) or 0) for r in records),
            'retry_calls':sum(int(r.get('retry_call_count',0) or 0) for r in records),
            'pages_recovered':sum(bool(r.get('page_recovery_triggered')) for r in records),
            'orientation_calls':sum(bool(r.get('orientation_raw_text')) for r in records),
            'pages_rotated':sum(bool(r.get('rotation_applied')) for r in records),
            'pages_deskewed':sum(bool(r.get('deskew_applied')) for r in records),
            'orientation_conflicts':sum(
                'ORIENTATION_VLM_VS_DETERMINISTIC_CONFLICT' in (r.get('quality_flags') or [])
                for r in records
            ),
            'qwen_calls_total':sum(
                len(r.get('classification_attempts') or [])
                + bool(r.get('orientation_raw_text'))
                + int(r.get('extraction_call_count',0) or 0)
                for r in records
            ),
            'extraction_passes':sum(len(r.get('extraction_strategies') or []) for r in records),
            'field_revisions':sum(len(r.get('field_revisions') or []) for r in records),
            'field_disagreements':sum(len(r.get('field_disagreements') or []) for r in records),
            'pages_confidence_high':sum(r.get('extraction_confidence_band')=='HIGH' for r in records),
            'pages_confidence_medium':sum(r.get('extraction_confidence_band')=='MEDIUM' for r in records),
            'pages_confidence_low':sum(r.get('extraction_confidence_band')=='LOW' for r in records),
            'tokens_in':int(tokens_in),'tokens_out':int(tokens_out),'tokens_total':int(tokens_in+tokens_out),
            'elapsed_s':elapsed,
        },
        'page_presence':page_presence,
        'page_records':[_json_safe_record(r) for r in records],
    }
    canonical_checkpoint_path(pdf_path).write_text(json.dumps(dossier,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    profiler_finish(OUTPUT_ROOT)
    return dossier

print('✅ Partie 1 V14 ROTATION_SHIFT_SAFE prête — orientation VLM + deskew avant extraction, prompts anti-décalage, recovery ciblé')


# Tests de régression V14 (hors GPU)

In [ ]:
# ============================================================================
# TESTS DE REGRESSION V14 — exécutés hors GPU, avant tout chargement modèle
# ============================================================================
import numpy as _np
from PIL import Image as _Img
import cv2 as _cv2

# --- 1) Schéma : 99 champs, hash contractuel Partie 2 inchangé --------------
_all_fields = [f for _dt, flds in FIELD_SCHEMA.items() for f in flds]
assert len(_all_fields) == 99, f"schema doit contenir 99 champs, trouvé {len(_all_fields)}"
assert len(set(_all_fields)) == 99, "doublon dans le schéma"
assert FIELD_SCHEMA_HASH == 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e', \
    "FIELD_SCHEMA_HASH modifié — CONTRAT PARTIE 2 ROMPU"
assert len(TTR_FIELDS) == 21 and len(TTR7_ACTIVE_FIELDS) == 7

# --- 2) Permis : structure complète NN-NNNNNNNN / NN-NN-NNNNNN --------------
assert PERMIT_FULL_RE.search('24-00013655/31-25-000591')
assert PERMIT_FULL_RE.search('2400013655/3125000591'.replace('2400013655','24-00013655').replace('3125000591','31-25-000591'))
assert not PERMIT_FULL_RE.search('24-00013655')          # une seule partie -> incomplet
assert not PERMIT_FULL_RE.search('0673417')              # numero manuscrit seul -> rejeté
assert not PERMIT_FULL_RE.search('24-00013655 / 31-25')  # tronqué -> rejeté
# _ttr7_valid : complet OK, partiel KO, absent KO
def _fake_ttr(permis):
    return {'raw_data':{'TTR_NUMERO_PERMIS':permis,'TTR_NOM':'YILDIRIM','TTR_PRENOM':'IBRAHIM',
                        'TTR_DATE_NAISSANCE':'1990-05-12','TTR_NATIONALITE':'TURQUE',
                        'TTR_DATE_DEBUT':'2025-01-01','TTR_DATE_FIN':'2025-12-31'},
            'ttr7_evidence':{'structure_dates_claire':True,'ancre_duree_trouvee':True,
                             'ancre_lieu_travail_trouvee':True},
            'doc_type':'TITRE_TRAVAIL','ttr_extraction_profile':TTR7_PROFILE}
assert _ttr7_valid(_fake_ttr('24-00013655 / 31-25-000591'))
assert not _ttr7_valid(_fake_ttr('24-00013655'))
assert not _ttr7_valid(_fake_ttr(None))

# --- 3) Orientation : parsing + detecteur deterministe synthetique ---------
assert _parse_orientation('{"rotation_clockwise": 90}') == 90
assert _parse_orientation('la page est tournee {"rotation_clockwise": 270} fin') == 270
assert _parse_orientation('0') == 0
assert _parse_orientation('45') == 0          # hors {0,90,180,270} -> 0 par securite
assert _parse_orientation('pas de json') == 0
# image synthetique : longues lignes horizontales -> UPRIGHT ; transposee -> QUARTER
_h = _np.full((600, 460), 255, dtype=_np.uint8)
for _y in range(40, 600, 55):
    _cv2.line(_h, (20, _y), (440, _y), 0, 2)
_img_h = _Img.fromarray(_h).convert('RGB')
assert detect_quarter_turn(_img_h)[0] == 'UPRIGHT'
assert detect_quarter_turn(_img_h.rotate(-90, expand=True, fillcolor='white'))[0] == 'QUARTER'

# --- 4) Deskew synthetique : +2.0 deg doivent etre corriges -----------------
_base = _np.full((800, 620), 255, dtype=_np.uint8)
for _y in range(60, 800, 60):
    _cv2.line(_base, (30, _y), (590, _y), 0, 2)
_img0 = _Img.fromarray(_base).convert('RGB')
_tilted = rotate_pil_expand_white(_img0, 2.0)
_img_fixed, _meta = deskew_for_vlm(_tilted)
assert _meta['deskew_applied'] is True, f"deskew non applique: {_meta}"
assert abs(abs(_meta['deskew_correction_angle_deg']) - 2.0) < 0.6, _meta
_img_same, _meta0 = deskew_for_vlm(_img0)
assert _meta0['deskew_applied'] is False or abs(_meta0.get('deskew_correction_angle_deg') or 0) < 0.35

# --- 5) finalize : TTR7 evalue sur 7 champs actifs uniquement ---------------
_fr = _fake_ttr('24-00013655 / 31-25-000591')
_fr.update({'extraction_attempts':[], 'field_revisions':[], 'field_disagreements':[],
            'quality_flags':[], 'extraction_call_count':1, 'retry_call_count':0,
            'page_recovery_triggered':False, 'classification_confidence':0.95,
            'extraction_tokens_in':0,'extraction_tokens_out':0,'extraction_elapsed_s':0.0,
            'extraction_strategies':[], 'field_candidates':{}, 'recovery_history':[],
            'semantic_issues_initial':[], 'extraction_taux_remplissage':1.0,
            'raw_data':{f: None for f in TTR_FIELDS} | _fr['raw_data']})
finalize_extraction_record(_fr)
assert _fr['extraction_taux_remplissage'] == 1.0, _fr['extraction_taux_remplissage']
assert not _fr['critical_fields_missing'], _fr['critical_fields_missing']
assert _fr['extraction_status'] == 'OK', _fr['extraction_status']
assert set(_fr['evaluated_fields']) == set(TTR7_ACTIVE_FIELDS)
assert len(_fr['unevaluated_fields']) == 14

# --- 6) Mapping compact : aller-retour alias -> canonique -------------------
_dt = 'ENGAGEMENT_DOMICILIATION'
_compact = {f'f{i+1:02d}': None for i, f in enumerate(CHAMPS_ATTENDUS[_dt])}
_full = expand_compact_extraction(_compact, _dt)
assert set(_full) == set(CHAMPS_ATTENDUS[_dt])

print('✅ TESTS V14 : 6/6 groupes passes (schema, permis, orientation, deskew, finalize TTR7, compact)')


# Exécution du pipeline (batch dossier)

In [ ]:
if not pdfs:
    print('⚠️ Aucun PDF dans', INPUT_DIR)
else:
    results=[]; errors=[]
    for i,pdf in enumerate(pdfs,1):
        print(f'\n[{i}/{len(pdfs)}] {pdf.name}')
        existing=load_existing_checkpoint(pdf)
        if existing is not None:
            print('↪ checkpoint RAW valide réutilisé')
            d=existing; statut='REPRIS'
        else:
            try:
                d=process_pdf(pdf); statut='TRAITE'
            except Exception as exc:
                log(f'❌ {pdf.name} : {repr(exc)}')
                errors.append({'source_file':pdf.name,'error':repr(exc),'date':datetime.now().isoformat(timespec='seconds')})
                continue
        s=d.get('stats') or {}
        results.append({'source_file':d.get('source_file'),'status':statut,'sha256':d.get('source_sha256'),
                        'pages':s.get('pages'),'page_records':s.get('page_records'),
                        'tokens_total':s.get('tokens_total'),'elapsed_s':s.get('elapsed_s'),
                        'qwen_calls_total':s.get('qwen_calls_total'),'extraction_calls':s.get('extraction_calls'),'retry_calls':s.get('retry_calls'),'pages_recovered':s.get('pages_recovered'),'pages_rotated':s.get('pages_rotated'),'pages_deskewed':s.get('pages_deskewed'),'orientation_calls':s.get('orientation_calls'),'pages_confidence_high':s.get('pages_confidence_high'),'pages_confidence_medium':s.get('pages_confidence_medium'),'pages_confidence_low':s.get('pages_confidence_low'),
                        'json_path':str(canonical_checkpoint_path(pdf))})
        print(f"✅ {statut} | pages={s.get('pages')} | Qwen calls={s.get('qwen_calls_total')} | extraction={s.get('extraction_calls')} | retries={s.get('retry_calls')} | recovered_pages={s.get('pages_recovered')} | tokens={s.get('tokens_total')} | temps={s.get('elapsed_s')}s")

    manifest={'schema_version':SCHEMA_VERSION,'pipeline_version':PIPELINE_VERSION,
              'field_schema_hash':FIELD_SCHEMA_HASH,'generated_at':datetime.now().isoformat(timespec='seconds'),
              'results':results,'errors':errors}
    MANIFEST_PATH.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(results).to_csv(INDEX_CSV_PATH,index=False,encoding='utf-8-sig')
    print('\n✅ Manifest :',MANIFEST_PATH)
    print('✅ Index    :',INDEX_CSV_PATH)
    print('✅ JSON RAW :',JSON_DIR)


# V14 ROTATION_SHIFT_SAFE — notes de mise en production

**Ce qui change par rapport à V13.8.7**

1. **Orientation générique (toutes pages, pas seulement TTR).**
   `resolve_orientations()` s'exécute après la classification et avant le gate :
   détection déterministe quart-de-tour (Hough, audit + cross-check), décision VLM
   batchée (0/90/180/270), rotation physique de l'image, puis **une seule**
   re-classification des pages tournées encore douteuses. Mode par défaut
   `ORIENTATION_MODE='VLM_ALL'` (qualité maximale, couvre le 180° que le
   déterministe ne peut pas distinguer) ; `'HYBRID'` disponible si le débit prime.

2. **Deskew (inclinaison fine 0.35°–7°).** Médiane des angles Hough
   quasi-horizontaux avec garde-fous (≥6 lignes, MAD≤1.8°). Correction par
   `warpAffine` INTER_CUBIC, fond blanc. Le couple (rotation, deskew) est
   **rejoué sur tout re-rendu HD** (`_render_for_strategy`) : initial TTR HD et
   recovery 2400 px partent du PDF brut mais arrivent au modèle redressés.

3. **Anti-décalage libellé/valeur (TTR & DOM).** Règle structurelle DOM :
   ordre vertical des valeurs confronté à la séquence fixe des libellés du
   formulaire. Prompt TTR7 fondé sur le squelette des champs. Nouveau filet
   `RECOVERY_ON_CRITICAL_SEMANTIC_MISMATCH` : une page décalée dont les champs
   critiques sont remplis mais sémantiquement invalides déclenche désormais la
   relecture page entière 2400 px.

4. **Permis TTR : structure complète obligatoire** `NN-NNNNNNNN / NN-NN-NNNNNN`
   (`PERMIT_FULL_RE` en `fullmatch` sur valeur compactée). Une moitié seule
   (ex. `24-00013655`) ou le numéro manuscrit seul → invalide → recovery.

5. **Recovery TTR ciblé** : `trigger_fields` = uniquement les champs réellement
   défaillants ; une valeur valide de la 1re lecture ne peut plus être écrasée
   silencieusement par la relecture.

**Invariant Partie 2** : 99 champs, noms inchangés, `FIELD_SCHEMA_HASH`
inchangé, aucune normalisation métier en Partie 1 (RAW + flags).

**Rappels benchmark** : comparer V13.8.7 vs V14 sur un lot contenant
(i) pages à 90/180/270°, (ii) scans inclinés 1–5°, (iii) TTR/DOM décalés.
Surveiller : `pages_rotated`, `pages_deskewed`, `orientation_conflicts`,
`ttr7_recovery_reason`, et le statut `PARTIELLE` vs `OK` par page.